# APC-Vis 7B — hoàn thành 375 lượt + 120 trace chọn lọc

Chọn **T4 ×2**, bật **Internet**, có thể thêm secret **HF_token**, rồi **Run All**.
Notebook đã nhúng checkpoint **1.011 lượt part 1** và kết quả **692 lượt part 2** tải ngày 12/11/2026.
Không cần gắn lại kết quả cũ. Chỉ chạy thêm **192 Left/Right + 183 Visibility**, không chạy lại Facing cho benchmark.

Sau đó replay **40 ảnh mỗi task**: Left/Right, Visibility, Object Facing; mỗi task **28 câu sai hợp lệ + 12 câu đúng theo lần chạy trước**.
Ưu tiên ba case cả năm model sai, APC bất đồng với model khác, đổi đáp án khi hoán vị lựa chọn, viewer từng được sửa và PCD chọn sai nhánh. Chọn tham lam có phạt lặp viewer/nhóm tín hiệu để đa dạng mẫu; mỗi task không trùng hash ảnh. Đây là điểm ưu tiên chẩn đoán từ kết quả đã có, không phải information gain đã đo thực nghiệm.
Replay ghi cả kết quả cũ/mới và thay đổi đáp án. Vì vậy số đúng/sai của lần replay có thể khác 28/12; không ép đáp án để đạt quota.
Ground truth chỉ dùng chọn mẫu chẩn đoán, không được đưa thêm vào prompt inference. Đây là tập chọn có chủ đích, không dùng accuracy của 120 câu để đại diện benchmark.

Benchmark và replay lưu riêng. Lỗi viewer vẫn được lưu; case lỗi sớm chỉ có trace tới bước hỏng, không giả tạo render.
Runtime/weights ở `/kaggle/temp`, kết quả ở `/kaggle/working/apc_completion/`.
Thời gian worker: tối đa 5 giờ cho phần còn thiếu, 3 giờ replay (không tính setup); mỗi phần có checkpoint để tiếp tục nếu bị ngắt.


In [ ]:
from pathlib import Path
import os,sys,json,subprocess,time,hashlib,shutil,base64,io,zipfile
WORK=Path('/kaggle/temp/apc_completion_runtime');WORK.mkdir(parents=True,exist_ok=True)
token=os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('HF_token')
    except Exception:
        token=None
if token:os.environ['HF_TOKEN']=token.strip()
else:print('HF_token chưa đặt; dùng quyền truy cập public. Có thể thêm Secret HF_token nếu bị giới hạn tải.')
del token
os.environ['TOKENIZERS_PARALLELISM']='false';os.environ['HF_HUB_DISABLE_TELEMETRY']='1';os.environ['HF_HUB_DISABLE_XET']='1'
SNAPSHOT_SHA='9059ed359baf89b49675ed3e0407f843b3ff28de78e5cae1c870adf79f44a339'
REPO=WORK/('repo_'+SNAPSHOT_SHA[:12]);REPO.mkdir(exist_ok=True)
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()==SNAPSHOT_SHA
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target=(REPO/name).resolve();target.relative_to(REPO.resolve())
    z.extractall(REPO)
SCRIPTS=WORK/'scripts';SCRIPTS.mkdir(exist_ok=True)
(SCRIPTS/'apc_common.py').write_text('"""Shared dataset/parser/state helpers; no model imports needed for offline validation."""\nimport hashlib,json,os,re,time,unicodedata\nfrom pathlib import Path\nPOLICY=\'strict_letter_or_unique_exact_choice_v1\'\ndef parse_option(text,choices):\n    def norm(x):return re.sub(r\'\\s+\',\' \',unicodedata.normalize(\'NFKC\',x).strip()).casefold()\n    match=re.fullmatch(r\'\\s*(?:(?:the\\s+)?answer\\s*(?:is|:)\\s*)?[\\(\\[]?([A-D])[\\)\\]]?[.!]?\\s*\',text,re.I)\n    strict=match.group(1).upper() if match and match.group(1).upper() in choices else None\n    exact=[k for k,v in choices.items() if norm(text) in (norm(v),norm(f\'{k}. {v}\'),norm(f\'{k}) {v}\'))]\n    return strict if strict else exact[0] if len(exact)==1 else None\n\ndef atomic_json(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(\'.tmp\')\n    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding=\'utf-8\');tmp.replace(path)\n\ndef read_jsonl(path,repair=False):\n    path=Path(path);rows={}\n    if not path.exists():return rows\n    with path.open(\'rb+\' if repair else \'rb\') as f:\n        while True:\n            start=f.tell();line=f.readline()\n            if not line:break\n            try:r=json.loads(line)\n            except (ValueError,UnicodeDecodeError):\n                if repair and not line.endswith(b\'\\n\') and not f.read():f.truncate(start);break\n                raise RuntimeError(\'Corrupt JSONL \'+str(path)+\' byte \'+str(start)) from None\n            rows[r[\'sample_id\']]=r\n            if repair and not line.endswith(b\'\\n\'):f.seek(0,2);f.write(b\'\\n\')\n    return rows\n\nPAPER_TASKS={\'orientation_on_the_left\':\'left_right\',\'orientation_in_front_of\':\'visibility\',\'multi_object_facing\':\'facing\'}\n\ndef paper_sample(row):\n    row=dict(row)\n    row[\'paper_task\']=PAPER_TASKS[row[\'category\']]\n    row[\'source_question\']=row[\'question\'];row[\'source_choices\']=dict(row[\'choices\'])\n    row[\'question_transform\']=\'unchanged\'\n    if row[\'category\']==\'orientation_in_front_of\':\n        match=re.fullmatch(r\'(.*?facing where it is facing, )is (.+?) in front of me or behind me\\?\',row[\'question\'],re.I)\n        if not match:raise ValueError(\'Unrecognized front/behind template for \'+row[\'sample_id\'])\n        mapping={\'in front of\':\'yes\',\'behind\':\'no\'}\n        if set(row[\'choices\'].values())!=set(mapping):raise ValueError(\'Unexpected front/behind options\')\n        row[\'question\']=match.group(1)+\'is \'+match.group(2)+\' visible or not?\'\n        row[\'choices\']={key:mapping[value] for key,value in row[\'choices\'].items()}\n        row[\'question_transform\']=\'front_behind_to_visibility_yes_no_v1\'\n        # Option positions and correct option letter are preserved, including each permutation.\n    return row\n\ndef choose_samples(rows,cfg):\n    scope=cfg.get(\'scope\')\n    if scope in (\'paper_tasks_original\',\'paper_tasks_visibility\'):\n        rows=[r for r in rows if r[\'category\'] in PAPER_TASKS]\n        if scope==\'paper_tasks_visibility\':rows=[paper_sample(r) for r in rows]\n    elif scope!=\'all_original\':raise ValueError(\'Unsupported scope\')\n    from apc_full_checkpoint import select_samples\n    return select_samples(rows,cfg)\n',encoding='utf-8')
(SCRIPTS/'apc_kaggle_patches.py').write_text('"""Apply auditable Kaggle compatibility fixes to a copied APC snapshot, not the checkout."""\nimport ast,hashlib,json,os\nfrom pathlib import Path\n\ndef patch_repo(repo,assets):\n    repo=Path(repo);assets=Path(assets);changes=[]\n    def edit(rel,fn):\n        p=repo/rel\n        backup=p.with_name(p.name+\'.apc_original\')\n        if not backup.exists():backup.write_text(p.read_text(),encoding=\'utf-8\')\n        old=backup.read_text();new=fn(old)\n        if new!=old:\n            ast.parse(new);p.write_text(new,encoding=\'utf-8\')\n            changes.append(dict(file=rel,before_sha=hashlib.sha256(old.encode()).hexdigest(),after_sha=hashlib.sha256(new.encode()).hexdigest()))\n    def detection(s):\n        s=s.replace(\'load_model(config_path, config.detection.ckpt_path)\',\'load_model(config_path, config.detection.ckpt_path, device=str(self.device))\')\n        needle=\'            text_threshold=self.config.detection.text_threshold,\\n\'\n        assert needle in s;s=s.replace(needle,needle+\'            device=str(self.device),\\n\')\n        # GroundingDINO\'s custom CUDA attention uses getCurrentCUDAStream()\n        # without a device guard. Match the current GPU to its input tensors.\n        tree=ast.parse(s)\n        method=next(n for n in ast.walk(tree) if isinstance(n,ast.FunctionDef) and n.name==\'run_detection\')\n        call=next(n for n in ast.walk(method) if isinstance(n,ast.Assign) and isinstance(n.value,ast.Call) and isinstance(n.value.func,ast.Name) and n.value.func.id==\'predict\')\n        lines=s.splitlines(keepends=True)\n        block=lines[call.lineno-1:call.end_lineno]\n        lines[call.lineno-1:call.end_lineno]=[\'        with torch.cuda.device(self.device):\\n\']+[\'    \'+line for line in block]\n        s=\'\'.join(lines)\n        return s\n    edit(\'apc/vision_modules/detection.py\',detection)\n    def orientation(s):\n        s=s.replace(\'from .src.orient_anything.render import render, Model\',\'\')\n        needle=\'        axis_model = Model(axis_obj_path, texture_filename=axis_texture_path)\'\n        s=s.replace(needle,\'        from .src.orient_anything.render import render, Model\\n\'+needle)\n        tree=ast.parse(s)\n        assignment=next(n for n in ast.walk(tree) if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id==\'orientation_ckpt_path\' for t in n.targets))\n        s=s.replace(ast.get_source_segment(s,assignment),\'orientation_ckpt_path = str(config.orientation.model_path)\',1)\n        s=s.replace(\'"facebook/dinov2-large",\',\'str(config.orientation.processor_path),\')\n        return s\n    edit(\'apc/vision_modules/orientation.py\',orientation)\n    def renderer(s):\n        s=s.replace("        if apc_args[\'render_whole_scene\']:","        z_trans = 0.0  # No backward shift needed when all objects are already in front.\\n        if apc_args[\'render_whole_scene\']:")\n        s=s.replace(\'        # Normalize positions\',\'        orientations = [orientations[i] for i in positions_ori_idx]\\n        colors = [colors[i] for i in positions_ori_idx]\\n\\n        # Normalize positions\')\n        s=s.replace(\'            xy_scale = grid_size / max_xy\',\'            xy_scale = grid_size / max(max_xy, 1e-6)\')\n        s=s.replace("            z_list = [pos[2] for pos in positions]\\n            max_z_obj_idx", "            if not positions:\\n                raise ValueError(\'No objects to render after excluding reference viewer\')\\n            z_list = [pos[2] for pos in positions]\\n            max_z_obj_idx",1)\n        return s\n    edit(\'apc/renderer.py\',renderer)\n    # Keep the mask statistics/geometry method; only remove a slow Python pixel loop.\n    def depth(s):\n        a=s.index(\'        depth_values = []\',s.index(\'def unproject_to_3D\'))\n        b=s.index(\'        # Find the most frequent depth\',a)\n        s=s[:a]+"        depth_values = depth_unproj[ys,xs]\\n        if depth_values.size == 0:\\n            raise ValueError(\'Empty segmentation mask; no 3D points\')\\n\\n"+s[b:]\n        a=s.index(\'        points_filtered = []\',s.index(\'def unproject_to_3D\'))\n        b=s.index(\'        # Compute final 3D coordinates\',a)\n        s=s[:a]+"        z=depth_unproj[ys,xs][depth_values_filtered]\\n        x=(xs[depth_values_filtered]-K[0,2])*z/K[0,0]\\n        y=(ys[depth_values_filtered]-K[1,2])*z/K[1,1]\\n        points_filtered=np.column_stack((x,-y,-z))\\n        if not len(points_filtered) or not np.isfinite(points_filtered).all():\\n            raise ValueError(\'Empty/nonfinite filtered 3D points\')\\n\\n"+s[b:]\n        return s\n    edit(\'apc/vision_modules/depth.py\',depth)\n    # External sources exist after HTTPS cloning.\n    edit(\'apc/vision_modules/src/orient_anything/paths.py\',lambda s:s.replace(\'"facebook/dinov2-large"\',repr(str(assets/\'dinov2_large\'))))\n    edit(\'apc/vision_modules/src/orient_anything/vision_tower.py\',lambda s:s.replace(\'bool_masked_pos: Optional[torch.Tensor] = None) -> torch.Tensor:\', \'bool_masked_pos: Optional[torch.Tensor] = None, **kwargs) -> torch.Tensor:\').replace(\'embeddings = self.dropout(embeddings)\',"embeddings = getattr(self, \'dropout\', torch.nn.Identity())(embeddings)"))\n    edit(\'apc/vision_modules/src/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py\',lambda s:s.replace(\'"bert-base-uncased"\',repr(str(assets/\'bert_base_uncased\'))).replace("\'bert-base-uncased\'",repr(str(assets/\'bert_base_uncased\'))))\n    return changes\n\nif __name__==\'__main__\':\n    import sys\n    changes=patch_repo(sys.argv[1],sys.argv[2]);Path(sys.argv[3]).write_text(json.dumps(changes,indent=2),encoding=\'utf-8\')\n',encoding='utf-8')
(SCRIPTS/'apc_install_native.py').write_text('"""Install native vision dependencies into the isolated Kaggle venv."""\nimport hashlib,json,os,shutil,subprocess,sys,time\nfrom pathlib import Path\nfrom apc_full_checkpoint import ensure_checkout\ncfg=json.loads(Path(sys.argv[1]).read_text());repo=Path(cfg[\'repo\']);work=Path(cfg[\'work\']);py=sys.executable\nearly_nvcc=shutil.which(\'nvcc\')\nif not early_nvcc:raise RuntimeError(\'nvcc is missing. Native APC vision setup needs a CUDA 12 development toolkit, not only an NVIDIA driver.\')\nearly_version=subprocess.check_output([early_nvcc,\'--version\'],text=True)\nif \'release 12.\' not in early_version:raise RuntimeError(\'Use a CUDA 12 toolkit with the cu124 Torch wheel. Found: \'+early_version)\nsrc=repo/\'apc/vision_modules/src\';src.mkdir(parents=True,exist_ok=True)\nlock_path=work/\'source_lock.json\';lock=json.loads(lock_path.read_text()) if lock_path.exists() else {}\nrepos={\n \'GroundingDINO\':(\'https://github.com/IDEA-Research/GroundingDINO.git\',\'main\'),\n \'ml-depth-pro\':(\'https://github.com/apple/ml-depth-pro.git\',\'main\'),\n \'orient_anything\':(\'https://github.com/SpatialVision/Orient-Anything.git\',\'main\'),\n \'omni3d\':(\'https://github.com/facebookresearch/omni3d.git\',\'main\'),\n \'detectron2\':(\'https://github.com/facebookresearch/detectron2.git\',\'main\'),\n \'pytorch3d\':(\'https://github.com/facebookresearch/pytorch3d.git\',\'V0.7.8\'),\n \'segment-anything\':(\'https://github.com/facebookresearch/segment-anything.git\',\'main\')}\ndef run(cmd,**kw):\n print(\'RUN\',cmd[:5],flush=True);subprocess.run(cmd,check=True,**kw)\nfor name,(url,ref) in repos.items():\n path=src/name\n if name in lock:ensure_checkout(path,url,lock[name][\'sha\'])\n if not (path/\'.git\').exists():\n  check=subprocess.run([\'git\',\'ls-remote\',\'--exit-code\',url,\'refs/heads/\'+ref,\'refs/tags/\'+ref],capture_output=True,text=True)\n  if check.returncode:\n   raise RuntimeError(\'Git ref missing or unreachable (case-sensitive): \'+ref+\' in \'+url+\'; \'+check.stderr.strip())\n  run([\'git\',\'clone\',\'--depth\',\'1\',\'--branch\',ref,url,str(path)])\n sha=subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'],cwd=path,text=True).strip()\n if name in lock and lock[name][\'sha\']!=sha:raise RuntimeError(\'Source revision changed: \'+name)\n lock[name]=dict(url=url,sha=sha,requested_ref=ref)\n lock_path.write_text(json.dumps(lock,indent=2),encoding=\'utf-8\')\n# Do not install Orient-Anything requirements: they downgrade transformers to 4.38.\npackages=[\'pip==25.0.1\',\'setuptools==75.3.0\',\'wheel\',\'ninja\',\'cython==0.29.37\',\'numpy==1.26.4\',\'scipy==1.11.4\',\n \'matplotlib==3.8.4\',\'python-box==7.3.2\',\'trimesh==4.7.4\',\'open3d==0.19.0\',\'pyglet==1.5.27\',\n \'opencv-python==4.10.0.84\',\'timm==1.0.22\',\'pillow-heif==0.18.0\',\'fvcore==0.1.5.post20221221\',\n \'iopath==0.1.10\',\'yacs\',\'cloudpickle\',\'termcolor\',\'pycocotools\',\'hydra-core==1.3.2\',\'omegaconf==2.3.0\',\n \'shapely==2.0.6\',\'rtree\',\'addict\',\'yapf==0.40.2\',\'supervision==0.22.0\',\'qwen-vl-utils==0.0.14\',\'av==13.1.0\']\nuv=cfg[\'uv\'];run([uv,\'pip\',\'install\',\'--python\',py,\'--no-cache\',*packages])\n# System display and compiler tools. No hardcoded DISPLAY=:1 from run_APC.py.\napt=[\'apt-get\'] if os.geteuid()==0 else [\'sudo\',\'-n\',\'apt-get\']\nrun(apt+[\'update\',\'-qq\']);run(apt+[\'install\',\'-y\',\'-qq\',\'xvfb\',\'xauth\',\'libgl1\',\'libglu1-mesa\',\'libgl1-mesa-dri\',\'build-essential\',\'ninja-build\'])\nimport torch\nnvcc=shutil.which(\'nvcc\')\nif not nvcc:raise RuntimeError(\'GroundingDINO CUDA build requires nvcc. Attach/select a Kaggle runtime with a CUDA 12 toolkit; driver alone is insufficient.\')\nversion=subprocess.check_output([nvcc,\'--version\'],text=True)\nif \'release 12.\' not in version:raise RuntimeError(\'CUDA toolkit must be major 12 for the cu124 wheel. Found: \'+version)\nenv=os.environ.copy();env[\'CUDA_HOME\']=str(Path(nvcc).resolve().parent.parent);env[\'FORCE_CUDA\']=\'1\';env[\'MAX_JOBS\']=\'2\'\nenv[\'TORCH_CUDA_ARCH_LIST\']=\';\'.join(sorted({\'.\'.join(map(str,torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())}))\n# Build with the already installed torch, without a second isolated copy of torch/CUDA.\nfor name in (\'segment-anything\',\'ml-depth-pro\',\'detectron2\',\'pytorch3d\',\'GroundingDINO\'):\n marker=work/(\'installed_\'+name+\'.json\')\n identity=dict(source_sha=lock[name][\'sha\'],torch=torch.__version__,python=sys.version.split()[0],cuda_compiler=version.strip())\n if marker.exists() and json.loads(marker.read_text())==identity:continue\n # A failed wheel build can leave its dist-info directory in bdist.*.\n # Remove packaging staging only, retaining compiled objects in build/temp.*.\n build_root=(src/name/\'build\').resolve()\n build_root.relative_to((src/name).resolve())\n if build_root.is_dir():\n  for staging in build_root.glob(\'bdist.*\'):\n   resolved=staging.resolve()\n   if staging.is_symlink() or resolved.parent!=build_root:\n    raise RuntimeError(\'Unexpected wheel staging path: \'+str(staging))\n   if resolved.is_dir():\n    print(\'Cleaning previous wheel staging:\',resolved,flush=True)\n    shutil.rmtree(resolved)\n run([uv,\'pip\',\'install\',\'--python\',py,\'--no-cache\',\'--no-build-isolation\',\'--no-deps\']+([\'-e\'] if name in (\'GroundingDINO\',\'ml-depth-pro\') else [])+[str(src/name)],env=env)\n marker.write_text(json.dumps(identity),encoding=\'utf-8\')\nprint(\'Native dependencies installed. Sources pinned in source_lock.json.\',flush=True)\n',encoding='utf-8')
(SCRIPTS/'apc_prepare_assets.py').write_text('"""Download once or reuse a read-only Kaggle asset input; never duplicate weight caches."""\nimport hashlib,json,os,shutil,sys,time,urllib.request\nfrom http.client import HTTPException\nfrom pathlib import Path\nfrom huggingface_hub import HfApi,hf_hub_download,snapshot_download\ncfg=json.loads(Path(sys.argv[1]).read_text());root=Path(cfg[\'asset_input\'] or cfg[\'asset_root\']);root.mkdir(parents=True,exist_ok=True)\nmanifest_file=root/\'assets_manifest.json\';old=json.loads(manifest_file.read_text()) if manifest_file.exists() else {}\nif cfg[\'asset_input\']:\n    required=[\'qwen_model/config.json\',\'qwen_model/preprocessor_config.json\',\'dinov2_large/config.json\',\'bert_base_uncased/config.json\',\n       \'checkpoints/sam_vit_h_4b8939.pth\',\'checkpoints/groundingdino_swint_ogc.pth\',\'checkpoints/depth_pro.pt\',\'checkpoints/orient_dino_weight.pt\']\n    missing=[s for s in required if not (root/s).is_file()]\n    for model in (\'qwen_model\',\'dinov2_large\',\'bert_base_uncased\'):\n        if not list((root/model).glob(\'*.safetensors\')):missing.append(model+\'/*.safetensors\')\n    if missing or not old:raise RuntimeError(\'ASSET_INPUT must be the prepared asset bundle root with assets_manifest.json. Missing \'+str(missing))\n    if old.get(\'model_id\')!=cfg[\'model_id\']:raise RuntimeError(\'Wrong Qwen checkpoint asset bundle.\')\n    if old.get(\'snapshots\',{}).get(\'qwen_model\',{}).get(\'sha\')!=cfg[\'model_revision\']:raise RuntimeError(\'Asset bundle has a different Qwen revision.\')\n    Path(sys.argv[2]).write_text(json.dumps(dict(old,root=str(root.resolve())),indent=2),encoding=\'utf-8\')\n    print(\'Using existing asset bundle:\',root);raise SystemExit(0)\napi=HfApi();snapshots={}\nmodels={\'qwen_model\':cfg[\'model_id\'],\'dinov2_large\':\'facebook/dinov2-large\',\'bert_base_uncased\':\'google-bert/bert-base-uncased\'}\nfor name,repo in models.items():\n    previous=old.get(\'snapshots\',{}).get(name,{})\n    revision=previous.get(\'sha\') if previous.get(\'repo\')==repo else cfg[\'model_revision\'] if name==\'qwen_model\' else \'main\'\n    info=api.model_info(repo,revision=revision,files_metadata=True)\n    if name==\'qwen_model\' and info.sha!=cfg[\'model_revision\']:raise RuntimeError(\'Cached Qwen revision differs from configuration.\')\n    wanted=[f for f in info.siblings if \'/\' not in f.rfilename and (f.rfilename.endswith((\'.json\',\'.safetensors\',\'.txt\',\'.model\',\'.jinja\')))]\n    remaining=sum(f.size or 0 for f in wanted if not (root/name/f.rfilename).is_file() or (root/name/f.rfilename).stat().st_size!=f.size)\n    if shutil.disk_usage(root).free<remaining+3*2**30:raise RuntimeError(\'Asset storage lacks space. Use a prepared Kaggle ASSET_INPUT; need \'+str(round(remaining/2**30,2))+\' GiB + 3 GiB reserve for \'+repo)\n    print(\'Download/cache\',repo,\'SHA\',info.sha,\'remaining GiB\',round(remaining/2**30,2),flush=True)\n    for f in sorted(wanted,key=lambda f:f.rfilename.endswith(\'.safetensors\')):\n        hf_hub_download(repo,f.rfilename,revision=info.sha,local_dir=str(root/name))\n        if f.size and (root/name/f.rfilename).stat().st_size!=f.size:raise RuntimeError(\'Truncated asset: \'+f.rfilename)\n    if not list((root/name).glob(\'*.safetensors\')):raise RuntimeError(\'No safetensors weights in \'+repo+\'; stop before model initialization.\')\n    snapshots[name]=dict(repo=repo,sha=info.sha)\nckpt=root/\'checkpoints\';ckpt.mkdir(exist_ok=True)\nurls={\n \'sam_vit_h_4b8939.pth\':\'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth\',\n \'groundingdino_swint_ogc.pth\':\'https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth\',\n \'depth_pro.pt\':\'https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt\'}\ndef download_checkpoint(url,target):\n    part=target.with_suffix(\'.part\')\n    for attempt in range(1,7):\n        try:\n            offset=part.stat().st_size if part.exists() else 0\n            headers={\'Accept-Encoding\':\'identity\'}\n            if offset:headers[\'Range\']=f\'bytes={offset}-\'\n            print(f\'Downloading {target.name}: attempt {attempt}/6, resume {offset/2**20:.0f} MiB\',flush=True)\n            with urllib.request.urlopen(urllib.request.Request(url,headers=headers),timeout=120) as response:\n                resumed=response.status==206\n                if resumed:\n                    content_range=response.headers.get(\'Content-Range\',\'\')\n                    if not content_range.startswith(f\'bytes {offset}-\'):raise OSError(\'Unexpected Content-Range: \'+content_range)\n                    total=int(content_range.split(\'/\')[-1])\n                else:\n                    offset=0 # Server ignored Range: overwrite, never append a full response.\n                    total=int(response.headers.get(\'Content-Length\',0))\n                if total and shutil.disk_usage(target.parent).free<total-offset+3*2**30:\n                    raise RuntimeError(\'Not enough disk for \'+target.name+\'; use ASSET_INPUT.\')\n                with part.open(\'ab\' if resumed else \'wb\') as f:\n                    while True:\n                        block=response.read(1024*1024)\n                        if not block:break\n                        f.write(block)\n                size=part.stat().st_size\n                if size<1000000 or total and size!=total:raise OSError(f\'Truncated download: {size}/{total} bytes\')\n            part.replace(target)\n            return\n        except (OSError,HTTPException,ValueError) as exc:\n            if attempt==6:raise RuntimeError(f\'Failed downloading {target.name}; partial file kept at {part}: {exc}\') from exc\n            print(f\'{target.name}: {type(exc).__name__}: {exc}; retrying...\',flush=True)\n            time.sleep(min(5*attempt,30))\n\nchecks={}\nfor name,url in urls.items():\n    target=ckpt/name;expected=old.get(\'files\',{}).get(name,{})\n    valid=target.is_file() and target.stat().st_size==expected.get(\'bytes\')\n    # Older runs wrote the manifest only after all downloads. Reuse their complete files.\n    if target.is_file() and not expected:\n        try:\n            with urllib.request.urlopen(urllib.request.Request(url,method=\'HEAD\'),timeout=60) as response:\n                valid=target.stat().st_size==int(response.headers.get(\'Content-Length\',0)) and target.stat().st_size>1000000\n        except OSError:pass\n    if not valid:\n        download_checkpoint(url,target)\n    else:print(\'Reusing\',name,flush=True)\n    digest=hashlib.sha256()\n    with target.open(\'rb\') as f:\n        for block in iter(lambda:f.read(16*1024*1024),b\'\'):digest.update(block)\n    if expected.get(\'sha256\') and digest.hexdigest()!=expected[\'sha256\']:raise RuntimeError(\'Asset checksum mismatch \'+name)\n    checks[name]=dict(url=url,bytes=target.stat().st_size,sha256=digest.hexdigest())\n    # Save completed checkpoint metadata even if a later download fails.\n    old.update(model_id=cfg[\'model_id\'],snapshots=snapshots,files=checks)\n    manifest_file.write_text(json.dumps(old,indent=2),encoding=\'utf-8\')\nori_revision=old.get(\'orientation_sha\',\'main\');ori_info=api.model_info(\'Viglong/Orient-Anything\',revision=ori_revision)\nori_path=hf_hub_download(\'Viglong/Orient-Anything\',\'croplargeEX2/dino_weight.pt\',revision=ori_info.sha,local_dir=str(root/\'orientation_download\'))\n# Symlink rather than make another full checkpoint copy.\ntarget=ckpt/\'orient_dino_weight.pt\'\nif target.is_symlink() and target.resolve()!=Path(ori_path).resolve():target.unlink()\nif not target.exists():target.symlink_to(Path(ori_path).resolve())\nmanifest=dict(model_id=cfg[\'model_id\'],snapshots=snapshots,orientation_sha=ori_info.sha,files=checks,root=str(root.resolve()))\nmanifest_file.write_text(json.dumps(manifest,indent=2),encoding=\'utf-8\');Path(sys.argv[2]).write_text(json.dumps(manifest,indent=2),encoding=\'utf-8\')\nprint(\'Assets ready:\',root,flush=True)\n',encoding='utf-8')
(SCRIPTS/'apc_qwen_adapter.py').write_text('"""Qwen 7B interface compatible with the APC multi-call pipeline."""\nimport time\nimport torch\nfrom transformers import Qwen2_5_VLForConditionalGeneration,AutoProcessor\nfrom qwen_vl_utils import process_vision_info\nfrom apc.vlms.vlm_base import VLMBase\n\nclass KaggleQwen(VLMBase):\n    def __init__(self,config,device=\'cuda:0\'):\n        super().__init__(device);self.config=config;self.device=device;self.calls=[];self.callback=None\n        self.load_model(config,device)\n    def load_model(self, config, device=\'cuda:0\'):\n        from pathlib import Path\n        import json\n        from apc_kaggle_memory import make_qwen_memory_plan, verify_loaded_partition\n\n        dtype = getattr(torch, config.dtype)\n        if config.dtype == \'bfloat16\' and not torch.cuda.is_bf16_supported(including_emulation=False):\n            raise RuntimeError(\'Native BF16 unavailable on T4; select float16.\')\n        report_dir = Path(config.memory_report_dir)\n        report_dir.mkdir(parents=True, exist_ok=True)\n        try:\n            device_map, report = make_qwen_memory_plan(\n                config.pretrained_model, dtype,\n                activation_reserve_gib=config.activation_reserve_gib,\n                vision_reserve_gib=config.vision_reserve_gib,\n            )\n        except Exception as exc:\n            (report_dir/\'memory_plan.json\').write_text(json.dumps(dict(error=str(exc)), indent=2))\n            raise\n        (report_dir/\'memory_plan.json\').write_text(json.dumps(report, indent=2))\n        print(\'Qwen memory plan:\', json.dumps({k:v for k,v in report.items() if k != \'device_map\'}), flush=True)\n        self.vlm_model, loading = Qwen2_5_VLForConditionalGeneration.from_pretrained(\n            config.pretrained_model, dtype=dtype, device_map=device_map,\n            attn_implementation=\'sdpa\', local_files_only=True,\n            low_cpu_mem_usage=True, output_loading_info=True,\n        )\n        if any(loading.get(k) for k in (\'missing_keys\',\'unexpected_keys\',\'mismatched_keys\',\'error_msgs\')):\n            raise RuntimeError(\'Qwen checkpoint did not load cleanly: \'+str(loading))\n        self.vlm_model.eval()\n        self.dtype = dtype\n        self.device = str(self.vlm_model.get_input_embeddings().weight.device)\n        actual_map = self.vlm_model.hf_device_map\n        (report_dir/\'model_device_map.json\').write_text(json.dumps(actual_map, default=str, indent=2))\n        if any(str(value) not in (\'0\',\'1\',\'cuda:0\',\'cuda:1\') for value in actual_map.values()):\n            raise RuntimeError(\'Qwen must remain entirely on the two GPUs. Inspect model_device_map.json.\')\n        if any(parameter.device.type == \'meta\' for parameter in self.vlm_model.parameters()):\n            raise RuntimeError(\'Qwen contains unloaded meta parameters; stop before inference.\')\n        placement = verify_loaded_partition(self.vlm_model, device_map)\n        placement[\'memory_allocated_gib\'] = [torch.cuda.memory_allocated(i)/2**30 for i in (0,1)]\n        (report_dir/\'qwen_placement.json\').write_text(json.dumps(placement, indent=2))\n        print(\'Qwen actual placement PASS:\', json.dumps(placement), flush=True)\n        self.processor = AutoProcessor.from_pretrained(\n            config.pretrained_model, local_files_only=True,\n            **{key:getattr(config,key) for key in (\'min_pixels\',\'max_pixels\') if getattr(config,key) is not None},\n        )\n    def generate_response(self,prompt,image=None,max_new_tokens=None,do_sample=False,temperature=0.0):\n        content=([{\'type\':\'image\',\'image\':image}] if image is not None else [])+[{\'type\':\'text\',\'text\':prompt}]\n        return self.process_messages([dict(role=\'user\',content=content)],max_new_tokens,do_sample,temperature)\n    def process_messages(self,messages,max_new_tokens=None,do_sample=False,temperature=0.0):\n        if getattr(self, \'before_generate\', None):self.before_generate()\n        started=time.monotonic()\n        import os,json\n        from pathlib import Path\n        folder=Path(os.environ[\'APC_TRACE_DIR\'])/\'vlm\'\n        folder.mkdir(exist_ok=True)\n        index=len(self.calls)+1\n        captured=[]\n        for message_index,message in enumerate(messages):\n            for content_index,content in enumerate(message.get(\'content\',[])):\n                if isinstance(content,dict) and content.get(\'type\')==\'image\':\n                    value=content[\'image\']\n                    image_file=folder/f\'{index:03d}_{message_index}_{content_index}.png\'\n                    if hasattr(value,\'save\'):\n                        value.save(image_file)\n                        captured.append(image_file.name)\n        # Save before generation so a failed call still has its prompt.\n\n        if self.callback:self.callback(\'vlm_call\',call=len(self.calls)+1)\n        text=self.processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)\n        has_image=any(isinstance(c,dict) and c.get(\'type\')==\'image\' for m in messages for c in m.get(\'content\',[]))\n        images,videos=process_vision_info(messages) if has_image else (None,None)\n        batch=self.processor(text=[text],images=images,videos=videos,padding=True,return_tensors=\'pt\')\n        length=int(batch[\'input_ids\'].shape[-1])\n        call_file=folder/f\'{index:03d}.json\'\n        call_file.write_text(json.dumps(dict(prompt=text,images=captured,status=\'started\',prompt_tokens=length),ensure_ascii=False))\n        if length>self.config.max_input_tokens:raise ValueError(\'APC intermediate prompt exceeds max_input_tokens; no silent truncation.\')\n        inputs={k:v.to(device=self.device,dtype=self.dtype) if k in (\'pixel_values\',\'pixel_values_videos\') else v.to(self.device) for k,v in batch.items()}\n        last_text=\'\\n\'.join(c.get(\'text\',\'\') for c in messages[-1].get(\'content\',[]) if c.get(\'type\')==\'text\')\n        is_final=last_text.lstrip().startswith(\'Here is the question and the response you provided:\')\n        cap=min(max_new_tokens or self.config.max_new_tokens,self.config.final_max_new_tokens if is_final else self.config.max_new_tokens)\n        tokenizer=self.processor.tokenizer;pad=tokenizer.pad_token_id\n        if pad is None:pad=tokenizer.eos_token_id\n        with torch.inference_mode():ids=self.vlm_model.generate(**inputs,do_sample=False,num_beams=1,max_new_tokens=cap,use_cache=True,pad_token_id=pad)\n        new=ids[:,length:];raw=self.processor.batch_decode(new,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0].strip()\n        call=dict(prompt_tokens=length,generated_tokens=int(new.shape[-1]),seconds=time.monotonic()-started,raw_output=raw)\n        call.update(images=captured,status=\'complete\')\n        call_file.write_text(json.dumps(dict(call,prompt=text),ensure_ascii=False))\n        self.calls.append(call)\n        if self.callback:self.callback(\'vlm_return\',call=len(self.calls),vlm_seconds=call[\'seconds\'])\n        return raw\n',encoding='utf-8')
(SCRIPTS/'apc_experiment.py').write_text('import contextlib,copy,hashlib,json,logging,os,random,shutil,signal,sys,time,traceback\n\nfrom pathlib import Path\n\nimport numpy as np\n\nimport torch\n\nfrom PIL import Image\n\nfrom box import Box\n\nfrom apc_common import atomic_json,parse_option,read_jsonl\n\nclass APCExperiment:\n    def __init__(self,cfg,assets,callback=None,sanity=False):\n        self.cfg=cfg;self.sanity=sanity;self.callback=callback or (lambda *a,**k:None)\n        import yaml\n        self.callback(\'loading_qwen\')\n        config=Box(yaml.safe_load(Path(\'apc/configs/qwenvl2_5_7b_instruct.yaml\').read_text()))\n        root=Path(assets[\'root\'])\n        config.vlm.pretrained_model=str(root/\'qwen_model\');config.vlm.dtype=cfg[\'dtype\']\n        config.vlm.min_pixels=cfg[\'min_pixels\'];config.vlm.max_pixels=cfg[\'max_pixels\']\n        config.vlm.memory_report_dir=cfg[\'work\']\n        config.vlm.activation_reserve_gib=cfg[\'vlm_activation_reserve_gib\']\n        config.vlm.vision_reserve_gib=cfg[\'vision_reserve_gib\']\n        config.vlm.final_max_new_tokens=cfg[\'final_max_new_tokens\']\n        config.vlm.max_new_tokens=cfg[\'intermediate_max_new_tokens\'];config.vlm.max_input_tokens=cfg[\'max_input_tokens\']\n        config.detection.ckpt_path=str(root/\'checkpoints/groundingdino_swint_ogc.pth\')\n        config.detection.box_threshold=cfg[\'detection_box_threshold\'];config.detection.text_threshold=cfg[\'detection_text_threshold\']\n        config.segmentation.ckpt_path=str(root/\'checkpoints/sam_vit_h_4b8939.pth\');config.depth.ckpt_path=str(root/\'checkpoints/depth_pro.pt\')\n        config.orientation.ckpt_path=str(root/\'hf_cache\');config.orientation.model_path=str(root/\'checkpoints/orient_dino_weight.pt\')\n        config.orientation.processor_path=str(root/\'dinov2_large\')\n        # Replace only the VLM class, retaining APC\'s stages and prompts.\n        from apc import apc_pipeline\n        from apc_qwen_adapter import KaggleQwen\n        apc_pipeline.get_vlm_model_class=lambda kind:KaggleQwen if kind==\'qwenvl2_5\' else (_ for _ in ()).throw(ValueError(kind))\n        self.callback(\'loading_apc_pipeline\')\n        vision_initial_device=\'cpu\' if cfg[\'vision_offload\'] else \'cuda:1\'\n        self.apc=apc_pipeline.APC(config,device_vlm=\'cuda:0\',device_vision=vision_initial_device)\n        self.vlm=self.apc.vlm_model;self.vlm.callback=self.callback\n        if cfg[\'vision_offload\']:\n            from apc_kaggle_memory import install_vision_scheduler\n            self.vision_scheduler=install_vision_scheduler(self.apc,self.vlm,\'cuda:1\')\n        memory={str(i):dict(free_gib=torch.cuda.mem_get_info(i)[0]/2**30, allocated_gib=torch.cuda.memory_allocated(i)/2**30) for i in (0,1)}\n        atomic_json(Path(cfg[\'work\'])/\'pipeline_memory.json\',memory)\n        print(\'APC loaded, GPU memory:\',json.dumps(memory),flush=True)\n        # Use the repository option-selection prompt.\n        self.apc.logger.setLevel(logging.INFO if sanity else logging.WARNING)\n        # Compute depth only once per sample even when several objects are abstracted.\n        original_depth=self.apc.depth_module.run_depth_estimation\n        self.depth_cache=None\n        def depth_once(image):\n            if self.depth_cache is None:\n                self.depth_cache=original_depth(image)\n                trace=Path(os.environ[\'APC_TRACE_DIR\'])\n                np.save(trace/\'depth.npy\',self.depth_cache)\n                finite=self.depth_cache[np.isfinite(self.depth_cache)]\n                low,high=np.percentile(finite,[2,98]) if finite.size else (0,1)\n                preview=np.clip((np.nan_to_num(self.depth_cache)-low)/max(high-low,1e-6),0,1)\n                Image.fromarray((preview*255).astype(\'uint8\')).save(trace/\'depth_preview.png\')\n                (trace/\'depth_metadata.json\').write_text(json.dumps(dict(shape=list(self.depth_cache.shape),preview_percentiles=[float(low),float(high)])))\n            return self.depth_cache\n        self.apc.depth_module.run_depth_estimation=depth_once\n        # SAM embedding is also identical for every object of one input image.\n        predictor=self.apc.detection_module.segmentation_predictor;original_set=predictor.set_image;self.sam_image_hash=None\n        def set_once(image,*a,**kw):\n            digest=hashlib.sha256(np.ascontiguousarray(image).tobytes()).hexdigest()\n            if digest!=self.sam_image_hash:\n                result=original_set(image,*a,**kw);self.sam_image_hash=digest;return result\n        predictor.set_image=set_once\n        for name in (\'get_objects_of_interest\',\'do_scene_abstraction\',\'get_reference_viewer\',\'do_perspective_change\',\'do_perspective_prompting_visual\'):\n            original=getattr(self.apc,name)\n            def timed(*a,_name=name,_fn=original,**kw):\n                self.callback(_name)\n                if _name==\'do_perspective_prompting_visual\' and len(a)>1:\n                    a=list(a);a[1]=copy.deepcopy(a[1]);a=tuple(a)\n                result=_fn(*a,**kw)\n                if _name==\'get_objects_of_interest\' and (not isinstance(result,tuple) or not result[0]):raise ValueError(\'APC could not parse object names\')\n                if _name==\'do_scene_abstraction\':\n                    if len(result)<2:raise ValueError(\'APC detected no usable objects\')\n                    for obj in result.values():\n                        if not np.isfinite(obj[\'position\']).all() or not np.isfinite(obj[\'orientation\']).all():raise FloatingPointError(\'Nonfinite scene geometry\')\n                trace=Path(os.environ[\'APC_TRACE_DIR\'])\n                (trace/f\'{_name}.json\').write_text(json.dumps(dict(result=result),default=lambda x:x.tolist() if isinstance(x,np.ndarray) else str(x),ensure_ascii=False))\n                return result\n            setattr(self.apc,name,timed)\n        original_render=self.apc.render_module.render_visual_prompt\n        def render_prompt(*args,**kw):\n            self.callback(\'render_visual_prompt\')\n            result=original_render(*args,**kw)\n            if True:result.save(Path(kw[\'trace_save_dir\'])/\'apc_visual_prompt.png\')\n            return result\n        self.apc.render_module.render_visual_prompt=render_prompt\n        if sanity:\n            def finite_hook(module,args,output):\n                logits=getattr(output,\'logits\',None)\n                if logits is not None and not torch.isfinite(logits[:,-1,:]).all().item():raise FloatingPointError(\'NaN/Inf in Qwen raw logits\')\n            self.vlm.vlm_model.register_forward_hook(finite_hook)\n        self.callback(\'ready\')\n\n    def run(self,row,mode,scratch):\n        self.depth_cache=None;self.sam_image_hash=None;self.vlm.calls=[]\n        options=\'\\n\'.join(f\'{k}. {v}\' for k,v in row[\'choices\'].items())\n        # Choices carry object descriptions for questions such as "Which object is higher?".\n        question=row[\'question\']+\'\\nAnswer options:\\n\'+options\n        with Image.open(row[\'image_path\']) as image:\n            image=image.convert(\'RGB\')\n            image.save(scratch/\'input.png\')\n            with torch.inference_mode():\n                if mode==\'direct\':raw=self.vlm.generate_response(question+\'\\nAnswer with the option letter only.\',image,max_new_tokens=self.cfg[\'final_max_new_tokens\'])\n                elif mode==\'apc_vis\':\n                    raw,_=self.apc.run_apc(image,question,trace_save_dir=str(scratch),perspective_prompt_type=\'visual\',\n                         visualize_trace=False,visualize_scene_abstraction=False,\n                         render_whole_scene=row[\'category\']==\'orientation_on_the_left\',return_conv_history=False,options=options)\n                else:raise ValueError(mode)\n        if not isinstance(raw,str):raise TypeError(\'APC response must be text\')\n        return raw\n',encoding='utf-8')
(SCRIPTS/'apc_prepare_data.py').write_text('"""Select APC sanity/full rows before decoding their images from the benchmark TSV."""\n\nimport base64\nfrom collections import Counter\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport re\nimport sys\n\nfrom apc_common import choose_samples\nfrom apc_kaggle_guardrails import atomic_json, check_resources, protocol_fingerprint\n\n\ndef metadata_from_tsv(tsv):\n    rows, references = [], {}\n    with Path(tsv).open(encoding="utf-8-sig", newline="") as stream:\n        reader = csv.DictReader(stream, delimiter="\\t")\n        required = {"index", "qid", "question", "image", "answer", "category"}\n        if not required.issubset(reader.fieldnames or []):\n            raise ValueError("Unexpected TSV schema: " + str(reader.fieldnames))\n        for source in reader:\n            sid, qid = str(source["index"]), str(source["qid"])\n            if sid in references:\n                raise ValueError("Duplicate sample ID: " + sid)\n            choices = {key: source[key] for key in "ABCD" if\n                       str(source.get(key, "")).strip().lower() not in ("", "none", "nan", "null")}\n            answer = source["answer"].strip().upper()\n            if len(choices) < 2 or answer not in choices:\n                raise ValueError("Invalid choices/answer: " + sid)\n            value = source["image"].strip()\n            references[sid] = value if re.fullmatch(r"\\d+", value) else None\n            rows.append(dict(sample_id=sid, question_id=qid, question=source["question"],\n                             choices=choices, answer=answer, category=source["category"],\n                             circular_group=re.sub(r"-[0-3]$", "", qid)))\n    return rows, references\n\n\ndef resolve_reference(sid, references):\n    seen = set()\n    while True:\n        if sid in seen or sid not in references:\n            raise ValueError("Cyclic/missing image reference: " + sid)\n        seen.add(sid)\n        if references[sid] is None:\n            return sid\n        sid = references[sid]\n\n\ndef materialize_images(tsv, rows, references, folder, config):\n    from PIL import Image\n    folder = Path(folder)\n    folder.mkdir(parents=True, exist_ok=True)\n    needed = {resolve_reference(row["sample_id"], references) for row in rows}\n    payloads = {}\n    with Path(tsv).open(encoding="utf-8-sig", newline="") as stream:\n        for source in csv.DictReader(stream, delimiter="\\t"):\n            sid = str(source["index"])\n            if sid not in needed:\n                continue\n            check_resources(folder, config)\n            value = source["image"].strip()\n            value = value.split(",", 1)[1] if value.startswith("data:image/") else value\n            raw = base64.b64decode(value, validate=True)\n            digest = hashlib.sha256(raw).hexdigest()\n            target = folder/(digest+".img")\n            if not target.exists() or target.stat().st_size != len(raw):\n                with Image.open(io.BytesIO(raw)) as image:\n                    image.verify()\n                temporary = target.with_suffix(".tmp")\n                temporary.write_bytes(raw)\n                temporary.replace(target)\n            payloads[sid] = (str(target), digest)\n            if len(payloads) == len(needed):\n                break\n    if set(payloads) != needed:\n        raise ValueError("Missing selected image payloads")\n    for row in rows:\n        row["image_path"], row["image_sha256"] = payloads[resolve_reference(row["sample_id"], references)]\n\n\ndef main(config_path):\n    from huggingface_hub import HfApi, hf_hub_download\n    config = json.loads(Path(config_path).read_text(encoding="utf-8"))\n    work, out = Path(config["work"]), Path(config["output"])\n    out.mkdir(parents=True, exist_ok=True)\n    csv.field_size_limit(2**31-1)\n    if config.get("dataset_input"):\n        tsv = Path(config["dataset_input"])\n        digest = hashlib.sha256()\n        with tsv.open("rb") as stream:\n            for block in iter(lambda: stream.read(16*2**20), b""):\n                digest.update(block)\n        data_sha = "local:"+digest.hexdigest()\n    else:\n        info = HfApi().dataset_info(config["dataset_id"], revision=config["data_revision"], files_metadata=True)\n        data_sha = info.sha\n        entry = next(item for item in info.siblings if item.rfilename == config["data_file"])\n        download = work/"data/tsv"\n        download.mkdir(parents=True, exist_ok=True)\n        target = download/config["data_file"]\n        required = 0 if target.exists() and target.stat().st_size == entry.size else entry.size or 0\n        from shutil import disk_usage\n        if disk_usage(work).free < required + config["disk_min_free_gib"]*2**30:\n            raise RuntimeError("Insufficient disk for TSV + reserve; set dataset_input to a mounted TSV.")\n        tsv = Path(hf_hub_download(config["dataset_id"], config["data_file"], repo_type="dataset",\n                                 revision=data_sha, local_dir=str(download)))\n    all_rows, references = metadata_from_tsv(tsv)\n    rows = choose_samples(all_rows, config)\n    if not rows:\n        raise ValueError("No selected rows")\n    materialize_images(tsv, rows, references, work/"data/selected_images", config)\n    assets = json.loads((work/"assets_ready.json").read_text(encoding="utf-8"))\n    manifest = dict(config=config, data_sha=data_sha, assets=assets,\n                    selected_rows=len(rows), total_dataset_rows=len(all_rows),\n                    circular_groups=len({row["circular_group"] for row in rows}),\n                    source_lock=json.loads((work/"source_lock.json").read_text(encoding="utf-8")),\n                    patches=json.loads((work/"patches.json").read_text(encoding="utf-8")),\n                    tasks=dict(Counter(row["paper_task"] for row in rows)),\n                    protocol="APC-Vis FP16; front/behind recast to visibility; strict exact-choice; no LLM judge")\n    manifest["fingerprint"] = protocol_fingerprint(manifest, rows)\n    previous = out/"manifest.json"\n    if previous.exists() and json.loads(previous.read_text())["fingerprint"] != manifest["fingerprint"]:\n        raise RuntimeError("Existing experiment has another protocol; choose a new experiment name.")\n    (out/"samples.jsonl").write_text("".join(json.dumps(row, ensure_ascii=False)+"\\n" for row in rows), encoding="utf-8")\n    atomic_json(previous, manifest)\n    print("Selected", len(rows), "rows:", manifest["tasks"], flush=True)\n\n\nif __name__ == "__main__":\n    main(sys.argv[1])\n',encoding='utf-8')
(SCRIPTS/'apc_kaggle_memory.py').write_text('"""Plan whole decoder layers for APC + FP16 Qwen on two Kaggle GPUs."""\n\nGIB = 2**30\n\n\ndef validate_partition_map(device_map, tensor_names=None):\n    """Reject mappings that make Accelerate recursively move a parent and its children."""\n    if not device_map or "" in device_map:\n        raise ValueError("A partitioned Qwen map must not contain the root key \'\'.")\n    names = list(device_map)\n    if any(device not in (0, 1) for device in device_map.values()):\n        raise ValueError("The Qwen partition must use only GPU 0 and GPU 1.")\n    for parent in names:\n        for child in names:\n            if child.startswith(parent + "."):\n                raise ValueError(f"Overlapping device-map entries: {parent!r} and {child!r}.")\n    if tensor_names is not None:\n        for name in tensor_names:\n            matches = [prefix for prefix in names if name == prefix or name.startswith(prefix + ".")]\n            if len(matches) != 1:\n                raise ValueError(f"Tensor {name!r} must have exactly one explicit device-map owner.")\n\n\ndef verify_loaded_partition(model, device_map):\n    """Check real tensor devices after dispatch, including nonpersistent buffers."""\n    tensors = list(model.named_parameters()) + list(model.named_buffers())\n    validate_partition_map(device_map, [name for name, _ in tensors])\n    byte_counts = [0, 0]\n    for name, tensor in tensors:\n        prefix = next(prefix for prefix in device_map if name == prefix or name.startswith(prefix + "."))\n        expected = device_map[prefix]\n        if tensor.device.type != "cuda" or tensor.device.index != expected:\n            raise RuntimeError(f"Tensor {name} is on {tensor.device}; expected cuda:{expected}.")\n        byte_counts[expected] += tensor.numel() * tensor.element_size()\n    return {"verified": True, "tensor_count": len(tensors),\n            "tensor_gib_by_gpu": [value / GIB for value in byte_counts]}\n\n\ndef plan_qwen_device_map(module_sizes, num_layers, free_bytes, *,\n                         text_prefix="model.language_model", visual_prefix="model.visual",\n                         activation_reserve_gib=(3.0, 2.0), vision_reserve_gib=6.0,\n                         tie_word_embeddings=False):\n    """Return an explicit GPU-only map, reserving GPU 1 for the APC vision stack.\n\n    Unlike an automatic balanced map, this does not equalize Qwen weights across\n    unequal GPU budgets. Decoder layers remain contiguous and never split.\n    module_sizes comes from accelerate.utils.modeling.compute_module_sizes.\n    """\n    if num_layers < 2 or len(free_bytes) != 2 or len(activation_reserve_gib) != 2:\n        raise ValueError("Expected at least two decoder layers and exactly two GPUs.")\n    reserves = [float(value) * GIB for value in activation_reserve_gib]\n    if min(reserves) < 0 or vision_reserve_gib < 0:\n        raise ValueError("Memory reserves cannot be negative.")\n    budgets = [int(free_bytes[0] - reserves[0]),\n               int(free_bytes[1] - reserves[1] - vision_reserve_gib * GIB)]\n    if min(budgets) <= 0:\n        raise RuntimeError("Insufficient free GPU memory after reserving APC/activation space. "\n                           "Stop old GPU workers or restart the Kaggle session.")\n    layers = [f"{text_prefix}.layers.{index}" for index in range(num_layers)]\n    embedding, norm = f"{text_prefix}.embed_tokens", f"{text_prefix}.norm"\n    for name in ("", visual_prefix, embedding, norm, *layers):\n        if name not in module_sizes:\n            raise ValueError(f"Qwen architecture/size table is missing {name!r}.")\n    if not tie_word_embeddings and "lm_head" not in module_sizes:\n        raise ValueError("Untied Qwen output head is missing from the size table.")\n    sizes = [int(module_sizes[name]) for name in layers]\n    head_size = int(module_sizes.get("lm_head", 0))\n    fixed_zero = int(module_sizes[visual_prefix] + module_sizes[embedding])\n    fixed_one = int(module_sizes[norm])\n    if tie_word_embeddings:\n        fixed_zero += head_size\n    else:\n        fixed_one += head_size\n    # The remaining small buffers belong to the explicitly mapped rotary module.\n    remainder = int(module_sizes[""]) - fixed_zero - fixed_one - sum(sizes)\n    if remainder < 0:\n        raise ValueError("Overlapping/inconsistent module sizes in the Qwen memory plan.")\n    fixed_zero += remainder\n    candidates = []\n    for cut in range(1, num_layers):\n        required = [fixed_zero + sum(sizes[:cut]), fixed_one + sum(sizes[cut:])]\n        if all(required[gpu] <= budgets[gpu] for gpu in (0, 1)):\n            candidates.append((cut, required))\n    if not candidates:\n        raise RuntimeError(\n            "FP16 Qwen + APC cannot fit the configured GPU reserves. "\n            f"Qwen={module_sizes[\'\']/GIB:.2f} GiB; weight budgets="\n            f"{budgets[0]/GIB:.2f}/{budgets[1]/GIB:.2f} GiB. "\n            "No CPU/disk offload or quantization was applied. Inspect memory_plan.json."\n        )\n    # Keep as much Qwen on GPU 0 as fits; GPU 1 needs space for vision inference.\n    cut, required = max(candidates, key=lambda candidate: candidate[0])\n    device_map = {visual_prefix: 0, embedding: 0,\n                  f"{text_prefix}.rotary_emb": 0, norm: 1,\n                  "lm_head": 0 if tie_word_embeddings else 1}\n    device_map.update({name: 0 if index < cut else 1 for index, name in enumerate(layers)})\n    validate_partition_map(device_map)\n    report = {\n        "strategy": "contiguous_nonoverlapping_decoder_layers_v2", "device_map": device_map,\n        "layers_gpu0": cut, "layers_gpu1": num_layers - cut,\n        "qwen_weights_gib": module_sizes[""] / GIB,\n        "free_before_qwen_gib": [value / GIB for value in free_bytes],\n        "weight_budget_gib": [value / GIB for value in budgets],\n        "planned_weights_gib": [value / GIB for value in required],\n        "activation_reserve_gib": list(activation_reserve_gib),\n        "vision_reserve_gpu1_gib": vision_reserve_gib,\n        "quantized": False, "cpu_or_disk_offload": False,\n    }\n    return device_map, report\n\n\ndef make_qwen_memory_plan(model_dir, dtype, *, activation_reserve_gib=(3.0, 2.0),\n                          vision_reserve_gib=6.0):\n    """Inspect the model on the meta device; allocate no real model weights."""\n    import gc\n    import torch\n    from accelerate import init_empty_weights\n    from accelerate.utils.modeling import check_device_map, compute_module_sizes\n    from transformers import AutoConfig, Qwen2_5_VLForConditionalGeneration\n\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError("Select Kaggle GPU T4 x2 before preparing the Qwen device map.")\n    config = AutoConfig.from_pretrained(model_dir, local_files_only=True)\n    config._attn_implementation = "sdpa"\n    with init_empty_weights(include_buffers=True):\n        skeleton = Qwen2_5_VLForConditionalGeneration(config)\n        skeleton.tie_weights()\n    if hasattr(skeleton.model, "language_model"):\n        text_prefix, visual_prefix = "model.language_model", "model.visual"\n    else:\n        text_prefix, visual_prefix = "model", "visual"\n    text_model = skeleton.get_submodule(text_prefix)\n    # Initialize each CUDA context before measuring usable memory.\n    for gpu in (0, 1):\n        with torch.cuda.device(gpu):\n            torch.empty(1, device=f"cuda:{gpu}")\n            torch.cuda.empty_cache()\n    free_bytes = [torch.cuda.mem_get_info(gpu)[0] for gpu in (0, 1)]\n    device_map, report = plan_qwen_device_map(\n        compute_module_sizes(skeleton, dtype=dtype), len(text_model.layers), free_bytes,\n        text_prefix=text_prefix, visual_prefix=visual_prefix,\n        activation_reserve_gib=activation_reserve_gib, vision_reserve_gib=vision_reserve_gib,\n        tie_word_embeddings=getattr(config.get_text_config(), "tie_word_embeddings", False),\n    )\n    validate_partition_map(device_map, [name for name, _ in skeleton.named_parameters()]\n                           + [name for name, _ in skeleton.named_buffers()])\n    check_device_map(skeleton, device_map)\n    del text_model, skeleton\n    gc.collect()\n    return device_map, report\n\n\nclass VisionScheduler:\n    """Keep one APC vision model on GPU 1 at a time, parking idle weights in RAM."""\n\n    def __init__(self, models, predictor, device="cuda:1", torch_module=None):\n        if torch_module is None:\n            import torch as torch_module\n        self.torch = torch_module\n        self.models = models\n        self.predictor = predictor\n        self.device = device\n        self.active = None\n\n    def park_all(self):\n        if self.active is None:\n            return\n        with self.torch.cuda.device(self.device):\n            if self.active == "sam" and self.torch.is_tensor(self.predictor.features):\n                self.predictor.features = self.predictor.features.to("cpu")\n            self.models[self.active].to("cpu")\n            self.active = None\n            self.torch.cuda.empty_cache()\n\n    def activate(self, name):\n        if name == self.active:\n            return\n        self.park_all()\n        with self.torch.cuda.device(self.device):\n            self.models[name].to(self.device)\n            if name == "sam" and self.torch.is_tensor(self.predictor.features):\n                self.predictor.features = self.predictor.features.to(self.device)\n        self.active = name\n\n\ndef install_vision_scheduler(apc, vlm, device="cuda:1"):\n    """Install scheduling after initializing the auxiliary models on CPU.\n\n    Model precision, inference, prompts, and thresholds are preserved. Moving the\n    SAM cached embedding with its model allows VLM refinement calls between masks.\n    """\n    from functools import wraps\n    from torchvision.transforms import Lambda\n\n    detection, depth, orientation = apc.detection_module, apc.depth_module, apc.orientation_module\n    predictor = detection.segmentation_predictor\n    models = {"grounding": detection.detection_model, "sam": detection.segmentation_model,\n              "depth": depth.depth_model, "orientation": orientation.orientation_model}\n    scheduler = VisionScheduler(models, predictor, device)\n    for module in (detection, depth, orientation):\n        module.device = device\n    apc.device_vision = device\n    apc.render_module.device = device\n    # Depth Pro closes over its initialization device in the preprocessing Lambda.\n    # Retain the original GPU normalization/dtype order when weights start on CPU.\n    transform = depth.depth_transform\n    if not hasattr(transform, "transforms") or len(transform.transforms) < 4:\n        raise RuntimeError("Unexpected Depth Pro preprocessing; inspect its upstream revision.")\n    transform.transforms[1] = Lambda(lambda tensor: tensor.to(device))\n\n    def wrap(owner, method_name, model_name):\n        method = getattr(owner, method_name)\n\n        @wraps(method)\n        def scheduled(*args, **kwargs):\n            scheduler.activate(model_name)\n            return method(*args, **kwargs)\n\n        setattr(owner, method_name, scheduled)\n\n    wrap(detection, "run_detection", "grounding")\n    for method in ("set_image", "predict", "predict_torch"):\n        wrap(predictor, method, "sam")\n    wrap(depth, "run_depth_estimation", "depth")\n    wrap(orientation, "get_3angle", "orientation")\n    vlm.before_generate = scheduler.park_all\n    return scheduler\n',encoding='utf-8')
(SCRIPTS/'apc_qwen_preflight.py').write_text('"""Check the real Accelerate dispatch path before compiling APC\'s native dependencies."""\n\nimport json\nimport sys\nfrom pathlib import Path\n\n\ndef probe_dispatch(device_map):\n    import gc\n    import torch\n    from accelerate import dispatch_model\n    from apc_kaggle_memory import validate_partition_map, verify_loaded_partition\n\n    validate_partition_map(device_map)\n\n    class Probe(torch.nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.order = list(device_map)\n            for name in self.order:\n                parent = self\n                parts = name.split(".")\n                for part in parts[:-1]:\n                    if not hasattr(parent, part):\n                        parent.add_module(part, torch.nn.Module())\n                    parent = getattr(parent, part)\n                leaf = torch.nn.Linear(4, 4)\n                with torch.no_grad():\n                    leaf.weight.copy_(torch.eye(4))\n                    leaf.bias.zero_()\n                leaf.register_buffer("probe_buffer", torch.ones(1), persistent=False)\n                parent.add_module(parts[-1], leaf)\n\n        def forward(self, value):\n            for name in self.order:\n                value = self.get_submodule(name)(value)\n            return value\n\n    # The same module paths/map as Qwen, but only a few KiB of actual weights.\n    model = dispatch_model(Probe().eval(), device_map=device_map)\n    result = verify_loaded_partition(model, device_map)\n    with torch.inference_mode():\n        value = torch.ones((1, 4), device="cuda:0")\n        output = model(value)\n        if not torch.allclose(output, value.to(output.device)):\n            raise RuntimeError("Accelerate cross-GPU dispatch probe returned an incorrect value.")\n    result.update(cross_gpu_forward=True, model="tiny_identity_probe", gpu_names=[\n        torch.cuda.get_device_name(index) for index in (0, 1)])\n    del model, value, output\n    gc.collect()\n    for index in (0, 1):\n        with torch.cuda.device(index):\n            torch.cuda.empty_cache()\n    return result\n\n\ndef main(config_path):\n    import torch\n    from huggingface_hub import hf_hub_download\n    from apc_kaggle_memory import make_qwen_memory_plan\n\n    config = json.loads(Path(config_path).read_text())\n    work = Path(config["work"])\n    model_dir = Path(config["asset_input"] or config["asset_root"]) / "qwen_model"\n    if not (model_dir / "config.json").is_file():\n        if config["asset_input"]:\n            raise RuntimeError("The mounted asset_input is missing qwen_model/config.json.")\n        hf_hub_download(config["model_id"], "config.json", revision=config["model_revision"],\n                        local_dir=str(model_dir))\n    device_map, report = make_qwen_memory_plan(\n        str(model_dir), getattr(torch, config["dtype"]),\n        activation_reserve_gib=config["vlm_activation_reserve_gib"],\n        vision_reserve_gib=config["vision_reserve_gib"],\n    )\n    report["dispatch_probe"] = probe_dispatch(device_map)\n    (work / "dispatch_preflight.json").write_text(json.dumps(report, indent=2))\n    print("PASS: Qwen map has no root/parent overlaps; real Accelerate dispatch and "\n          "a tiny forward succeeded on both GPUs.", flush=True)\n    print(json.dumps({key: value for key, value in report.items() if key != "device_map"}), flush=True)\n\n\nif __name__ == "__main__":\n    main(sys.argv[1])\n',encoding='utf-8')
(SCRIPTS/'apc_kaggle_guardrails.py').write_text('"""Small, dependency-free controls shared by the APC notebook and worker."""\n\nfrom collections import Counter, defaultdict\nfrom contextlib import contextmanager\nimport errno\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport time\nimport traceback\nimport zipfile\n\nGIB = 2**30\nTASKS = ("left_right", "visibility", "facing")\nOPERATIONAL_KEYS = {\n    "work", "repo", "output", "experiment", "uv", "asset_root", "asset_input",\n    "dataset_input", "retry_failed", "resume_input", "sample_timeout_seconds",\n    "load_timeout_seconds", "run_timeout_seconds", "disk_min_free_gib",\n    "trace_max_total_mib", "trace_max_sample_mib", "trace_keep_per_task",\n    "max_consecutive_errors", "max_total_errors", "archive_max_mib",\n}\n\n\nclass ResourceLimitError(RuntimeError):\n    pass\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    with temporary.open("w", encoding="utf-8") as stream:\n        json.dump(value, stream, ensure_ascii=False, indent=2)\n        stream.flush()\n        os.fsync(stream.fileno())\n    temporary.replace(path)\n\n\ndef read_records(path, repair=False):\n    """Repair only an interrupted final record, never an interior corruption."""\n    path = Path(path)\n    records = {}\n    if not path.exists():\n        return records\n    with path.open("rb+" if repair else "rb") as stream:\n        while True:\n            start = stream.tell()\n            line = stream.readline()\n            if not line:\n                break\n            try:\n                record = json.loads(line)\n            except (ValueError, UnicodeDecodeError):\n                if repair and not line.endswith(b"\\n") and not stream.read():\n                    stream.truncate(start)\n                    break\n                raise RuntimeError(f"Corrupt JSONL at {path}:{start}") from None\n            records[str(record["sample_id"])] = record\n            if repair and not line.endswith(b"\\n"):\n                stream.seek(0, 2)\n                stream.write(b"\\n")\n    return records\n\n\ndef protocol_fingerprint(manifest, rows):\n    config = {k: v for k, v in manifest["config"].items() if k not in OPERATIONAL_KEYS}\n    samples = [{k: v for k, v in row.items() if k != "image_path"} for row in rows]\n    assets = {k: v for k, v in manifest.get("assets", {}).items() if k != "root"}\n    identity = dict(config=config, samples=samples, data_sha=manifest["data_sha"],\n                    assets=assets, source_lock=manifest.get("source_lock"),\n                    patches=manifest.get("patches"))\n    return hashlib.sha256(json.dumps(identity, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n\n\ndef select_samples(rows, config):\n    """Input has already received the paper task/question transformation."""\n    tasks = config.get("paper_tasks", list(TASKS))\n    rows = [row for row in rows if row.get("paper_task") in tasks]\n    if config.get("run_kind") == "sanity":\n        by_task = {task: {} for task in tasks}\n        for row in rows:\n            by_task[row["paper_task"]].setdefault(row["circular_group"], []).append(row)\n        selected = []\n        for task in tasks:\n            groups = list(by_task[task].values())[:config.get("sanity_groups_per_task", 1)]\n            if not groups or any(len(group) != 2 for group in groups):\n                raise ValueError(f"Sanity needs complete 2-row circular groups for {task}")\n            selected.append([row for group in groups for row in group])\n        # Exercise all three code paths before the second option permutation.\n        return [part[index] for index in range(max(map(len, selected)))\n                for part in selected if index < len(part)]\n    if config.get("limit") is not None:\n        groups = {row["circular_group"] for row in rows[:config["limit"]]}\n        rows = [row for row in rows if row["circular_group"] in groups]\n    return rows\n\n\ndef tree_bytes(folder):\n    return sum(path.stat().st_size for path in Path(folder).rglob("*")\n               if path.is_file() and not path.is_symlink())\n\n\ndef check_resources(folder, config, trace=None):\n    free = shutil.disk_usage(folder).free\n    reserve = int(config["disk_min_free_gib"] * GIB)\n    if free < reserve:\n        raise ResourceLimitError(f"Disk low: {free/GIB:.2f} GiB free; reserve {reserve/GIB:.2f} GiB. "\n                                 "Stopped before filling the filesystem.")\n    if trace is not None and tree_bytes(trace) > config["trace_max_sample_mib"] * 2**20:\n        raise ResourceLimitError("This sample exceeded trace_max_sample_mib; inspect its trace.")\n    return free / GIB\n\n\ndef remove_trace(path, trace_root):\n    """Only delete a sample child of this run\'s explicit trace root."""\n    path, trace_root = Path(path), Path(trace_root).resolve()\n    resolved = path.resolve()\n    resolved.relative_to(trace_root)\n    if resolved == trace_root or path.is_symlink():\n        raise ValueError("Refusing to delete the trace root or a symlink")\n    if resolved.exists():\n        shutil.rmtree(resolved)\n\n\ndef retain_trace(trace, root, row, record, config, kept):\n    task = row["paper_task"]\n    wanted = config["run_kind"] == "sanity" or record["status"] != "ok" or kept[task] < config["trace_keep_per_task"]\n    fits = tree_bytes(root) <= config["trace_max_total_mib"] * 2**20\n    if wanted and fits:\n        kept[task] += 1\n        record["trace_path"] = str(Path(trace).relative_to(Path(root).parent))\n        record["trace_bytes"] = tree_bytes(trace)\n    else:\n        remove_trace(trace, root)\n        record["trace_path"] = None\n        record["trace_dropped"] = "budget" if wanted else "retention_policy"\n\n\nclass FailureBudget:\n    def __init__(self, config):\n        self.config = config\n        self.consecutive = self.total = 0\n        self.identical = Counter()\n\n    def observe(self, record):\n        # Local failures are still persisted/scored, but do not stop full runs.\n        if self.config["run_kind"] == "full" and record.get("error_type") in (\n                "ReferenceViewerError", "InvalidAnswer"):\n            self.consecutive = 0\n            return None\n        if record["status"] == "ok":\n            self.consecutive = 0\n            return None\n        self.consecutive += 1\n        self.total += 1\n        signature = (record.get("error_type", record["status"]), record.get("message", "")[:300])\n        self.identical[signature] += 1\n        if self.consecutive >= self.config["max_consecutive_errors"]:\n            return "consecutive_errors"\n        if self.identical[signature] >= self.config["max_consecutive_errors"]:\n            return "repeated_error"\n        if self.total >= self.config["max_total_errors"]:\n            return "total_errors"\n        return None\n\n\ndef fatal_exception(exc):\n    if isinstance(exc, (ResourceLimitError, NameError, AttributeError, ImportError, TypeError, AssertionError)):\n        return True\n    if isinstance(exc, MemoryError) or (isinstance(exc, OSError) and exc.errno in\n            (errno.ENOSPC, errno.EDQUOT, errno.EIO)):\n        return True\n    text = str(exc).lower()\n    return isinstance(exc, RuntimeError) and any(word in text for word in (\n        "out of memory", "device-side assert", "illegal memory", "cuda error", "cublas_status"))\n\n\ndef successful(record):\n    return record.get("status") == "ok" and record.get("parse_valid") is True\n\n\ndef pending_rows(rows, records, fingerprint, retry_failed):\n    allowed = {str(row["sample_id"]) for row in rows}\n    for sid, record in records.items():\n        if sid not in allowed or record.get("fingerprint") != fingerprint or record.get("mode") != "apc_vis":\n            raise RuntimeError("Resume mismatch: dataset/protocol/code differs from the checkpoint")\n    return [row for row in rows if str(row["sample_id"]) not in records or\n            retry_failed and not successful(records[str(row["sample_id"])])]\n\n\ndef make_report(rows, records, stop_reason=None):\n    tasks, groups = defaultdict(list), defaultdict(list)\n    for row in rows:\n        tasks[row["paper_task"]].append(row)\n        groups[row["circular_group"]].append(row)\n    valid = [record for record in records.values() if successful(record)]\n    def hit(row):\n        record = records.get(str(row["sample_id"]), {})\n        return successful(record) and record.get("prediction") == row["answer"]\n    by_task = {}\n    for task, samples in tasks.items():\n        attempts = [records[str(row["sample_id"])] for row in samples if str(row["sample_id"]) in records]\n        good = [r for r in attempts if successful(r)]\n        by_task[task] = dict(expected=len(samples), attempted=len(attempts), valid=len(good),\n                            status="PASS" if len(good) == len(samples) else "FAIL" if attempts else "NOT_RUN",\n                            mean_seconds=sum(r["seconds"] for r in good)/len(good) if good else None)\n    passed = len(valid) == len(rows) and not stop_reason\n    return dict(technical_status="PASS" if passed else "FAIL", expected_rows=len(rows),\n                attempted_rows=len(records), valid_predictions=len(valid),\n                pipeline_errors=sum(r["status"] == "error" for r in records.values()),\n                invalid_outputs=sum(r["status"] == "invalid" for r in records.values()),\n                missing_rows=len(rows)-len(records), stop_reason=stop_reason, by_task=by_task,\n                row_accuracy=sum(map(hit, rows))/len(rows),\n                circular_accuracy=sum(all(hit(row) for row in group) for group in groups.values())/len(groups),\n                accuracy_note="Sanity accuracy is diagnostic only; PASS tests execution, not correctness of every answer.")\n\n\ndef results_archive(out, config, debug=False):\n    """Bound uncompressed bytes before writing. Never archive the weights/venv/images."""\n    out = Path(out)\n    archive = out / ("sanity_traces.zip" if debug else "results.zip")\n    candidates = []\n    for path in out.rglob("*"):\n        if not path.is_file() or path.is_symlink():\n            continue\n        if path.name == "archive_status.json":\n            continue  # Written after this export; never pack the previous attempt\'s status.\n        relative = path.relative_to(out)\n        if "trace" in relative.parts:\n            if debug:\n                candidates.append(path)\n        elif not debug and path.suffix in (".json", ".jsonl", ".txt", ".log"):\n            candidates.append(path)\n    total = sum(path.stat().st_size for path in candidates)\n    maximum = int((config["trace_max_total_mib"] if debug else config["archive_max_mib"]) * 2**20)\n    free = shutil.disk_usage(out).free\n    reserve = config["disk_min_free_gib"] * GIB\n    if not candidates or total > maximum or free < reserve + total + 2**20:\n        archive.unlink(missing_ok=True)  # Do not offer a stale ZIP from a previous attempt.\n        return dict(status="SKIPPED", reason="empty, byte budget or free disk", source_bytes=total)\n    temporary = archive.with_suffix(".zip.tmp")\n    try:\n        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as zipped:\n            for path in candidates:\n                zipped.write(path, path.relative_to(out))\n        temporary.replace(archive)\n    except BaseException:\n        temporary.unlink(missing_ok=True)\n        archive.unlink(missing_ok=True)\n        raise\n    return dict(status="OK", path=str(archive), bytes=archive.stat().st_size, source_bytes=total)\n\n\ndef export_safely(out, config, debug=False):\n    """An export failure must never hide the worker/setup exception."""\n    status = {}\n    for kind in ([False, True] if debug else [False]):\n        key = "traces" if kind else "results"\n        try:\n            status[key] = results_archive(out, config, debug=kind)\n        except Exception as exc:\n            status[key] = dict(status="FAILED", error_type=type(exc).__name__, message=str(exc))\n    try:\n        atomic_json(Path(out)/"archive_status.json", status)\n    except OSError:\n        pass\n    print("Archive:", json.dumps(status), flush=True)\n    return status\n\n\ndef terminate_worker(child):\n    if child.poll() is None:\n        child.terminate()\n    try:\n        child.wait(timeout=15)\n    except subprocess.TimeoutExpired:\n        child.kill()\n        child.wait()\n\n\n@contextmanager\ndef setup_stage(name, out, config):\n    started = time.monotonic()\n    path = Path(out)/"setup_status.json"\n    atomic_json(path, dict(stage=name, status="RUNNING"))\n    try:\n        yield\n    except BaseException as exc:\n        try:\n            atomic_json(path, dict(stage=name, status="FAIL", seconds=time.monotonic()-started,\n                                  error_type=type(exc).__name__, message=str(exc), traceback=traceback.format_exc()))\n        except OSError:\n            pass\n        try:\n            from apc_launcher import collect_reports\n            collect_reports(config["work"], out)\n        except Exception:\n            pass\n        export_safely(out, config)\n        print(f"Setup failed in {name}; inspect {path}", flush=True)\n        try:\n            from apc_launcher import show_downloads\n            show_downloads(out)\n        except Exception:\n            pass\n        raise\n    else:\n        atomic_json(path, dict(stage=name, status="PASS", seconds=time.monotonic()-started))\n\n\ndef run_logged(name, command, out, *, env, cwd=None):\n    folder = Path(out)/"setup_logs"\n    folder.mkdir(parents=True, exist_ok=True)\n    log_path = folder/(name+".log")\n    started = time.monotonic()\n    with log_path.open("a", encoding="utf-8") as stream:\n        child = subprocess.Popen(command, env=env, cwd=cwd, stdout=stream, stderr=subprocess.STDOUT)\n        try:\n            next_update = 0\n            while child.poll() is None:\n                elapsed = time.monotonic()-started\n                if elapsed >= next_update:\n                    print(f"{name}: {elapsed/60:.1f} min; log={log_path}", flush=True)\n                    next_update = elapsed+20\n                time.sleep(1)\n        except BaseException:\n            terminate_worker(child)\n            raise\n    if child.returncode:\n        with log_path.open("rb") as stream:\n            stream.seek(max(0, log_path.stat().st_size-16000))\n            print(stream.read().decode("utf-8", errors="replace"))\n        raise RuntimeError(f"{name} failed, exit={child.returncode}; see {log_path}")\n    print(f"{name}: PASS ({time.monotonic()-started:.1f}s)", flush=True)\n',encoding='utf-8')
(SCRIPTS/'apc_trace_patches.py').write_text('"""Apply trace edits to the copied APC source after its compatibility patches."""\n\nimport ast\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\n\n\nEDITS = {\n    "apc/apc_pipeline.py": [\n        ("import logging", "import logging\\nimport json\\nfrom pathlib import Path"),\n        ("for obj_name in objects_of_interest:\\n            self.logger.info",\n         "for obj_index,obj_name in enumerate(objects_of_interest):\\n"\n         "            trace=Path(apc_args[\'trace_save_dir\'])\\n"\n         "            prefix=trace / f\'object_{obj_index}\'\\n"\n         "            prefix.with_suffix(\'.name\').write_text(obj_name)\\n            self.logger.info"),\n        ("if len(boxes) == 0:",\n         "np.save(str(prefix)+\'_boxes.npy\', np.asarray([np.asarray(b.cpu() if hasattr(b,\'cpu\') else b) for b in boxes]))\\n"\n         "            if len(boxes) == 0:"),\n        ("# largest mask",\n         "# largest mask\\n"\n         "            Image.fromarray(np.asarray(segment_mask.cpu() if hasattr(segment_mask,\'cpu\') else segment_mask).astype(\'uint8\')*255).save(str(prefix)+\'_mask.png\')\\n"\n         "            np.save(str(prefix)+\'_selected_box.npy\',np.asarray(box2D))"),\n        ("return abstract_scene_tgt",\n         "Path(os.environ[\'APC_TRACE_DIR\'],\'scene.json\').write_text(json.dumps(dict(viewer=ref_viewer,camera=abstract_scene_src,transformed=abstract_scene_tgt),default=lambda x:np.asarray(x).tolist()))\\n"\n         "        return abstract_scene_tgt"),\n    ],\n    "apc/vision_modules/detection.py": [\n        ("        boxes_sorted = sort_by_scores",\n         "        from pathlib import Path\\n        import json\\n"\n         "        with (Path(os.environ[\'APC_TRACE_DIR\'])/\'detector.jsonl\').open(\'a\') as stream:\\n"\n         "            stream.write(json.dumps(dict(category=category,boxes=boxes.detach().cpu().tolist(),scores=scores.detach().cpu().tolist()))+\'\\\\n\')\\n"\n         "        boxes_sorted = sort_by_scores"),\n    ],\n    "apc/vision_modules/orientation.py": [\n        ("        # Obtain each angle",\n         "        from pathlib import Path\\n        import json\\n"\n         "        folder=Path(apc_args[\'trace_save_dir\'])\\n"\n         "        record={\'category\':category,\'bbox\':list(bbox) if bbox is not None else None,\'angles\':np.asarray(pred_angles).tolist()}\\n"\n         "        index=len(list(folder.glob(\'orientation_input_*.png\')))\\n"\n         "        image.save(folder/f\'orientation_input_{index:02d}.png\')\\n"\n         "        # Obtain each angle"),\n        ("        direction_vector = self.orientation_to_direction(azimuth_deg, polar_deg)",\n         "        direction_vector = self.orientation_to_direction(azimuth_deg, polar_deg)\\n"\n         "        record[\'direction_vector\']=np.asarray(direction_vector).tolist()\\n"\n         "        (folder/f\'orientation_angles_{index:02d}.json\').write_text(json.dumps(record,default=lambda x:np.asarray(x).tolist()))"),\n    ],\n}\n\n\ndef apply_patches(repo):\n    changes = []\n    for name, edits in EDITS.items():\n        path = Path(repo)/name\n        source = path.read_text(encoding="utf-8")\n        original = source\n        for before, after in edits:\n            if after in source:\n                continue\n            if source.count(before) != 1:\n                raise ValueError(f"Expected exactly one trace patch target in {name}: {before[:100]}")\n            source = source.replace(before, after, 1)\n        ast.parse(source)\n        path.write_text(source, encoding="utf-8")\n        changes.append(dict(file=name, before_sha=hashlib.sha256(original.encode()).hexdigest(),\n                            after_sha=hashlib.sha256(source.encode()).hexdigest()))\n    return changes\n\n\nif __name__ == "__main__":\n    changes = apply_patches(sys.argv[1])\n    Path(sys.argv[2]).write_text(json.dumps(changes, indent=2), encoding="utf-8")\n',encoding='utf-8')
(SCRIPTS/'apc_worker.py').write_text('"""Use the GPU-tested v4 executor with strict viewers and resumable full-run reports."""\n\nimport json\nfrom pathlib import Path\nimport sys\n\nfrom apc_kaggle_guardrails import ResourceLimitError, atomic_json\nfrom apc_full_checkpoint import write_full_report\nimport apc_v4_worker\n\n\ndef full_experiment(config, assets, callback, sanity=False):\n    from apc_experiment import APCExperiment\n    import torch\n    experiment = APCExperiment(config, assets, callback, sanity=False)\n    # The same numerical check used by the successful six-row sanity, without\n    # turning on verbose INFO logging for thousands of rows.\n    def finite_logits(module, args, output):\n        logits = getattr(output, "logits", None)\n        if logits is not None and not torch.isfinite(logits[:, -1, :]).all().item():\n            raise FloatingPointError("NaN/Inf in Qwen raw logits")\n    experiment.vlm.vlm_model.register_forward_hook(finite_logits)\n    original = experiment.run\n    def run(row, mode, trace):\n        try:\n            return original(row, mode, trace)\n        finally:\n            # Full prompts/images remain in retained debug traces. Keep the\n            # checkpoint compact enough to download and mount in the next session.\n            calls = [{key:value for key,value in call.items() if key in\n                      ("prompt_tokens", "generated_tokens", "seconds", "status")}\n                     for call in experiment.vlm.calls]\n            audit = Path(trace)/"reference_viewer.json"\n            if audit.is_file():\n                value = json.loads(audit.read_text())\n                calls.append(dict(reference_viewer_strategy=value["strategy"],\n                                  resolved_viewer=value["resolved_viewer"]))\n            experiment.vlm.calls = calls\n    experiment.run = run\n    return experiment\n\n\ndef validate_full_trace(trace):\n    warnings = apc_v4_worker.validate_trace(trace)\n    audit = Path(trace)/"reference_viewer.json"\n    if not audit.is_file():\n        raise AssertionError("Missing reference viewer audit")\n    value = json.loads(audit.read_text())\n    if not value.get("resolved_viewer") or value["strategy"] == "unresolved":\n        raise ValueError("Unresolved reference viewer")\n    if value["strategy"] == "explicit_camera_question":\n        warnings = [warning for warning in warnings if "fell back to camera" not in warning]\n    return warnings\n\n\ndef run_full(manifest_path, retry_failed=None, runner=None):\n    manifest = json.loads(Path(manifest_path).read_text())\n    out = Path(manifest["config"]["output"])\n    failure, paused = None, False\n    try:\n        (runner or apc_v4_worker.run_manifest)(manifest_path, retry_failed=retry_failed,\n            experiment_factory=full_experiment, evidence_validator=validate_full_trace)\n    except ResourceLimitError as exc:\n        if str(exc) == "Run time budget reached; pending rows remain in the report.":\n            paused = True\n            source = out/"run_failure.json"\n            if source.exists():\n                source.replace(out/"pause_event.json")\n            state_file = out/"run/state.json"\n            if state_file.exists():\n                state = json.loads(state_file.read_text())\n                atomic_json(state_file, dict(state, stage="paused", stop_reason="session_time_budget"))\n        else:\n            failure = exc\n    except RuntimeError as exc:\n        # v4 requires all six sanity rows to pass. A full benchmark may finish\n        # with explicit row errors; its score still includes those rows as wrong.\n        if not str(exc).startswith("Sanity FAIL: inspect sanity_report.json"):\n            failure = exc\n    except BaseException as exc:\n        failure = exc\n    try:\n        component = out/"sanity_report.json"\n        if component.exists():\n            destination = out/"runtime_reports/worker_report.json"\n            destination.parent.mkdir(exist_ok=True)\n            component.replace(destination)\n        report = write_full_report(out, pause=paused, failure=failure)\n    except Exception:\n        if failure is not None:\n            raise failure\n        raise\n    print("APC FULL", report["completion_status"],\n          f"{report[\'attempted_rows\']}/{report[\'expected_rows\']}",\n          "valid=", report["valid_predictions"], "errors=", report["pipeline_errors"],\n          "invalid=", report["invalid_outputs"], flush=True)\n    if failure is not None:\n        raise failure\n    return report\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--manifest", required=True)\n    group = parser.add_mutually_exclusive_group()\n    group.add_argument("--retry-failed", action="store_true", dest="retry_failed")\n    group.add_argument("--no-retry-failed", action="store_false", dest="retry_failed")\n    parser.set_defaults(retry_failed=None)\n    args = parser.parse_args()\n    run_full(args.manifest, args.retry_failed)\n',encoding='utf-8')
(SCRIPTS/'apc_launcher.py').write_text('"""Supervise a time-bounded full run and export an independent resume checkpoint."""\n\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport time\n\nfrom apc_kaggle_guardrails import atomic_json, terminate_worker\nfrom apc_full_checkpoint import prepare_resume,export_full, write_full_report\nfrom apc_v4_launcher import collect_reports as collect_base\n\n\ndef collect_reports(work, out):\n    collect_base(work, out)\n    source = Path(work)/"reference_patches.json"\n    if source.is_file():\n        shutil.copy2(source, Path(out)/"runtime_reports/reference_patches.json")\n\n\ndef show_downloads(out):\n    from IPython.display import FileLink, display\n    for name in ("full_report.json", "summary.json", "checkpoint.zip", "results.zip", "traces.zip",\n                 "checkpoint_status.json", "archive_status.json", "setup_status.json", "run_failure.json",\n                 "run/predictions_apc_vis.jsonl", "run/worker.log"):\n        path = Path(out)/name\n        if path.is_file():\n            display(FileLink(str(path)))\n\n\ndef launch(config, python, scripts, env, retry_failed=None):\n    out = Path(config["output"])\n    folder = out/"run"\n    folder.mkdir(exist_ok=True)\n    state_path = folder/"state.json"\n    state_path.unlink(missing_ok=True)\n    atomic_json(out/"full_report.json", dict(completion_status="RUNNING"))\n    command = [str(python), "-u", str(Path(scripts)/"apc_worker.py"), "--manifest", str(out/"manifest.json")]\n    if retry_failed is not None:\n        command.append("--retry-failed" if retry_failed else "--no-retry-failed")\n    started, failure = time.monotonic(), None\n    with (folder/"worker.log").open("a", encoding="utf-8") as log:\n        child = subprocess.Popen(command, env=env, cwd=config["repo"], stdout=log, stderr=subprocess.STDOUT)\n        try:\n            while child.poll() is None:\n                try:\n                    state = json.loads(state_path.read_text()) if state_path.exists() else {}\n                except (OSError, ValueError):\n                    state = {}\n                elapsed = time.monotonic()-started\n                count, total = state.get("processed", 0), state.get("total", 0)\n                rate = state.get("new_processed", 0)/max(1, state.get("elapsed_seconds", 1))\n                eta = round(max(0, total-count)/rate/3600, 2) if rate else "?"\n                print("FULL", state.get("pipeline_stage", state.get("stage", "starting")),\n                      state.get("paper_task", ""), state.get("sample_id", ""), f"{count}/{total or \'?\'}",\n                      f"{100*count/total:.1f}%" if total else "",\n                      "new errors=", state.get("errors", 0), "new invalid=", state.get("invalid", 0),\n                      "elapsed h=", round(elapsed/3600, 2), "ETA h=", eta,\n                      "free GiB=", state.get("disk_free_gib"), flush=True)\n                sample_start = state.get("sample_started_unix")\n                if sample_start and time.time()-sample_start > config["sample_timeout_seconds"]+120:\n                    raise TimeoutError("Full sample watchdog exceeded timeout + 120s")\n                if not state.get("new_processed") and not sample_start and elapsed > config["load_timeout_seconds"]+120:\n                    raise TimeoutError("Full model initialization watchdog exceeded timeout")\n                # Allow the worker to finish its current bounded sample, write the\n                # partial report and exit normally at the next loop boundary.\n                if elapsed > config["run_timeout_seconds"]+config["sample_timeout_seconds"]+120:\n                    raise TimeoutError("Full session watchdog exceeded its export reserve")\n                time.sleep(20)\n            if child.returncode:\n                with (folder/"worker.log").open("rb") as stream:\n                    stream.seek(max(0, (folder/"worker.log").stat().st_size-16000))\n                    print(stream.read().decode("utf-8", errors="replace"))\n                raise RuntimeError(f"Full worker failed (exit {child.returncode}); inspect checkpoint and worker.log")\n        except BaseException as exc:\n            failure = exc\n        finally:\n            terminate_worker(child)\n    if failure:\n        try:\n            existing = json.loads((out/"full_report.json").read_text())\n            if existing.get("completion_status") == "ERROR":\n                report = dict(existing, supervisor_error=str(failure))\n                atomic_json(out/"full_report.json", report)\n                atomic_json(out/"summary.json", report)\n            else:\n                report = write_full_report(out, failure=failure)\n        except Exception as report_error:\n            print("Report update failed:", repr(report_error))\n    try:\n        collect_reports(config["work"], out)\n    except OSError as exc:\n        print("Report collection:", repr(exc))\n    try:\n        export_full(out, config)\n    except Exception as exc:\n        print("Export failed:", repr(exc), "; raw files remain in Output")\n    show_downloads(out)\n    if failure:\n        raise failure\n    report = json.loads((out/"full_report.json").read_text())\n    print("FULL", report["completion_status"], report["attempted_rows"], "/", report["expected_rows"],\n          "technical=", report["technical_status"], flush=True)\n    return report\n',encoding='utf-8')
(SCRIPTS/'apc_v4_worker.py').write_text('"""Run APC with durable checkpoints, bounded traces and explicit technical coverage."""\n\nfrom collections import Counter\nimport functools\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport signal\nimport sys\nimport time\nimport traceback\n\nfrom apc_common import parse_option\nfrom apc_kaggle_guardrails import (\n    FailureBudget, ResourceLimitError, atomic_json, check_resources, fatal_exception,\n    make_report, pending_rows, read_records, remove_trace, retain_trace, successful,\n)\n\nPIPELINE_STAGES = {\n    "get_objects_of_interest", "do_scene_abstraction", "get_reference_viewer",\n    "do_perspective_change", "do_perspective_prompting_visual", "render_visual_prompt",\n    "detection", "segmentation", "depth", "unprojection", "orientation",\n}\n\n\ndef attach_stages(experiment, callback):\n    for owner, method, stage in (\n        (experiment.apc.detection_module, "run_detection", "detection"),\n        (experiment.apc.detection_module, "run_segmentation", "segmentation"),\n        (experiment.apc.depth_module, "run_depth_estimation", "depth"),\n        (experiment.apc.depth_module, "unproject_to_3D", "unprojection"),\n        (experiment.apc.orientation_module, "run_orientation_estimation", "orientation"),\n    ):\n        original = getattr(owner, method)\n        @functools.wraps(original)\n        def measured(*args, _original=original, _stage=stage, **kwargs):\n            callback(_stage)\n            return _original(*args, **kwargs)\n        setattr(owner, method, measured)\n\n\ndef validate_trace(trace):\n    trace = Path(trace)\n    required = ("input.png", "depth.npy", "scene.json", "apc_visual_prompt.png")\n    missing = [name for name in required if not (trace/name).is_file()]\n    orientations = list(trace.glob("orientation_angles_*.json"))\n    masks = list(trace.glob("object_*_mask.png"))\n    if missing or not orientations or not masks:\n        raise ValueError(f"Incomplete APC evidence: missing={missing}, orientations={len(orientations)}, masks={len(masks)}")\n    for path in orientations:\n        vector = json.loads(path.read_text())["direction_vector"]\n        if len(vector) != 3 or not all(math.isfinite(float(x)) for x in vector):\n            raise FloatingPointError("Invalid orientation vector in " + path.name)\n    # Real model output must include a decodable rendered image, not just model loading.\n    from PIL import Image\n    with Image.open(trace/"apc_visual_prompt.png") as image:\n        image.verify()\n    scene = json.loads((trace/"scene.json").read_text())\n    warnings = []\n    if scene.get("viewer") == "camera":\n        warnings.append("Reference viewer fell back to camera; inspect the VLM reference-viewer response.")\n    if len([name for name in scene.get("camera", {}) if name != "camera"]) < 2:\n        warnings.append("Fewer than two objects survived scene abstraction; inspect detection/refinement.")\n    return warnings\n\n\ndef run_manifest(manifest_path, retry_failed=None, experiment_factory=None, torch_module=None,\n                 numpy_module=None, evidence_validator=validate_trace):\n    if torch_module is None:\n        import torch as torch_module\n    if numpy_module is None:\n        import numpy as numpy_module\n    torch, np = torch_module, numpy_module\n    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))\n    config = manifest["config"]\n    out = Path(config["output"])\n    folder = out/"run"\n    folder.mkdir(parents=True, exist_ok=True)\n    rows = [json.loads(line) for line in (out/"samples.jsonl").read_text(encoding="utf-8").splitlines()]\n    predictions = folder/"predictions_apc_vis.jsonl"\n    if config.get("resume_input") and not predictions.exists():\n        source = Path(config["resume_input"])\n        previous_manifest = json.loads((source/"manifest.json").read_text())\n        if previous_manifest["fingerprint"] != manifest["fingerprint"]:\n            raise RuntimeError("resume_input has a different protocol fingerprint")\n        previous = source/"run/predictions_apc_vis.jsonl"\n        if previous.is_file():\n            import shutil\n            shutil.copy2(previous, predictions)\n    records = read_records(predictions, repair=True)\n    retry_failed = config["retry_failed"] if retry_failed is None else retry_failed\n    pending = pending_rows(rows, records, manifest["fingerprint"], retry_failed)\n    previous_failure = out/"run_failure.json"\n    if previous_failure.exists():\n        history = out/"failure_history"\n        history.mkdir(exist_ok=True)\n        previous_failure.replace(history/f"run_{time.time_ns()}.json")\n    state = dict(stage="starting", processed=len(rows)-len(pending), total=len(rows),\n                 new_processed=0, errors=0, invalid=0, sample_started_unix=None)\n    started = time.monotonic()\n    current_trace, events = None, []\n    stop_reason, failure = None, None\n    kept = Counter()\n    for row in rows:\n        record = records.get(str(row["sample_id"]), {})\n        if record.get("trace_path"):\n            kept[row["paper_task"]] += 1\n\n    def update(stage, **values):\n        state.update(stage=stage, elapsed_seconds=time.monotonic()-started, updated_unix=time.time(), **values)\n        if stage in PIPELINE_STAGES:\n            state["pipeline_stage"] = stage\n            if len(events) < 512:\n                events.append(dict(stage=stage, seconds=state["elapsed_seconds"]))\n        state["disk_free_gib"] = check_resources(out, config, current_trace)\n        atomic_json(folder/"state.json", state)\n\n    try:\n        update("initializing")\n        if pending:\n            if experiment_factory is None:\n                from apc_experiment import APCExperiment\n                experiment_factory = APCExperiment\n            assets = json.loads((Path(config["work"])/"assets_ready.json").read_text())\n            experiment = experiment_factory(config, assets, update, sanity=config["run_kind"] == "sanity")\n            if hasattr(experiment, "apc"):\n                attach_stages(experiment, update)\n            budget = FailureBudget(config)\n            with predictions.open("a", encoding="utf-8") as stream:\n                for row in pending:\n                    if time.monotonic()-started > config["run_timeout_seconds"]:\n                        raise ResourceLimitError("Run time budget reached; pending rows remain in the report.")\n                    sid = str(row["sample_id"])\n                    trace_root = folder/"trace"\n                    current_trace = trace_root/"apc_vis"/sid\n                    # The TSV uses decimal IDs. Refuse paths outside this run.\n                    if not sid.isdecimal():\n                        raise ValueError("Unsafe/non-decimal sample ID: " + sid)\n                    remove_trace(current_trace, trace_root)\n                    current_trace.mkdir(parents=True)\n                    os.environ["APC_TRACE_DIR"] = str(current_trace)\n                    events.clear()\n                    state["pipeline_stage"] = "sample_start"\n                    update("sample_start", sample_id=sid, paper_task=row["paper_task"], sample_started_unix=time.time())\n                    seed = int.from_bytes(hashlib.sha256(f"{config[\'seed\']}:{sid}".encode()).digest()[:4], "big")\n                    random.seed(seed)\n                    np.random.seed(seed)\n                    torch.manual_seed(seed)\n                    torch.cuda.manual_seed_all(seed)\n                    for gpu in (0, 1):\n                        torch.cuda.reset_peak_memory_stats(gpu)\n                    record = dict(sample_id=sid, paper_task=row["paper_task"], fingerprint=manifest["fingerprint"],\n                                  mode="apc_vis", status="error", prediction=None, parse_valid=False, raw_output="", seed=seed)\n                    fatal = None\n                    sample_start = time.monotonic()\n                    handler = None\n                    def timeout(signum, frame):\n                        raise TimeoutError("APC sample exceeded sample_timeout_seconds")\n                    if hasattr(signal, "SIGALRM"):\n                        handler = signal.signal(signal.SIGALRM, timeout)\n                        signal.alarm(config["sample_timeout_seconds"])\n                    try:\n                        raw = experiment.run(row, "apc_vis", current_trace)\n                        prediction = parse_option(raw, row["choices"])\n                        warnings = evidence_validator(current_trace)\n                        record.update(status="ok" if prediction is not None else "invalid", raw_output=raw,\n                                      prediction=prediction, parse_valid=prediction is not None, quality_warnings=warnings,\n                                      correct=prediction == row["answer"])\n                        if prediction is None:\n                            record.update(error_type="InvalidAnswer", message="Response is not an exact answer option")\n                            state["invalid"] += 1\n                    except Exception as exc:\n                        state["errors"] += 1\n                        record.update(error_type=type(exc).__name__, message=str(exc),\n                                      traceback=traceback.format_exc(), failed_stage=state.get("pipeline_stage"),\n                                      failed_operation=state.get("stage"))\n                        traceback.print_exc()\n                        if fatal_exception(exc):\n                            fatal = exc\n                    finally:\n                        if hasattr(signal, "SIGALRM"):\n                            signal.alarm(0)\n                            signal.signal(signal.SIGALRM, handler)\n                        record.update(seconds=time.monotonic()-sample_start, stages=list(events),\n                                      vlm_calls=getattr(getattr(experiment, "vlm", None), "calls", []))\n                        try:\n                            record["peak_vram_gib"] = {str(i):torch.cuda.max_memory_allocated(i)/2**30 for i in (0, 1)}\n                        except Exception:\n                            record["peak_vram_gib"] = None\n                        # Release excess debug files before writing durable, small results.\n                        try:\n                            retain_trace(current_trace, trace_root, row, record, config, kept)\n                        except Exception as trace_error:\n                            record["trace_management_error"] = repr(trace_error)\n                            fatal = fatal or trace_error\n                        stream.write(json.dumps(record, ensure_ascii=False)+"\\n")\n                        stream.flush()\n                        os.fsync(stream.fileno())\n                        records[sid] = record\n                    current_trace = None\n                    state["processed"] += 1\n                    state["new_processed"] += 1\n                    update("sample_done", last_status=record["status"], last_seconds=record["seconds"], sample_started_unix=None)\n                    atomic_json(out/"sanity_report.json", make_report(rows, records))\n                    print(json.dumps(dict(sample_id=sid, task=row["paper_task"], status=record["status"],\n                                          prediction=record["prediction"], seconds=round(record["seconds"], 2))), flush=True)\n                    reason = budget.observe(record)\n                    if fatal:\n                        raise fatal\n                    if reason:\n                        raise RuntimeError("Stopped by failure budget: " + reason)\n        update("complete", sample_started_unix=None)\n    except BaseException as exc:\n        failure = exc\n        stop_reason = type(exc).__name__+": "+str(exc)\n        try:\n            atomic_json(out/"run_failure.json", dict(error_type=type(exc).__name__, message=str(exc),\n                        traceback=traceback.format_exc(), pipeline_stage=state.get("pipeline_stage")))\n            atomic_json(folder/"state.json", dict(state, stage="error", stop_reason=stop_reason))\n        except OSError:\n            pass\n    report = make_report(rows, records, stop_reason)\n    report["wall_seconds"] = time.monotonic()-started\n    report["new_attempts"] = state["new_processed"]\n    report["reused_valid_rows"] = sum(successful(r) for r in records.values()) - sum(\n        successful(records.get(str(row["sample_id"]), {})) for row in pending)\n    report["quality_warnings"] = {sid:r["quality_warnings"] for sid,r in records.items() if r.get("quality_warnings")}\n    if all(task in report["by_task"] and report["by_task"][task]["mean_seconds"] is not None for task in ("left_right", "visibility", "facing")):\n        means = {task:report["by_task"][task]["mean_seconds"] for task in report["by_task"]}\n        report["rough_full_inference_hours"] = dict(\n            part1=(698*means["left_right"]+688*means["visibility"])/3600,\n            part2=692*means["facing"]/3600,\n            note="Tiny-sample extrapolation; excludes setup/load/export and is not a time guarantee.")\n    try:\n        atomic_json(out/"sanity_report.json", report)\n    except OSError:\n        if failure is None:\n            raise\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    if failure:\n        raise failure\n    if report["technical_status"] != "PASS":\n        raise RuntimeError("Sanity FAIL: inspect sanity_report.json and run/predictions_apc_vis.jsonl")\n    return report\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--manifest", required=True)\n    retry = parser.add_mutually_exclusive_group()\n    retry.add_argument("--retry-failed", dest="retry_failed", action="store_true")\n    retry.add_argument("--no-retry-failed", dest="retry_failed", action="store_false")\n    parser.set_defaults(retry_failed=None)\n    arguments = parser.parse_args()\n    run_manifest(arguments.manifest, retry_failed=arguments.retry_failed)\n',encoding='utf-8')
(SCRIPTS/'apc_v4_launcher.py').write_text('"""Notebook-side supervision; stop the worker before exporting its checkpoints."""\n\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport time\n\nfrom apc_kaggle_guardrails import atomic_json, export_safely, make_report, read_records, terminate_worker\n\n\ndef collect_reports(work, out):\n    destination = Path(out)/"runtime_reports"\n    destination.mkdir(exist_ok=True)\n    for name in ("source_lock.json", "patches.json", "trace_patches.json", "assets_ready.json",\n                 "dispatch_preflight.json", "memory_plan.json", "model_device_map.json",\n                 "qwen_placement.json", "pipeline_memory.json", "gpu_check.txt"):\n        source = Path(work)/name\n        if source.is_file():\n            shutil.copy2(source, destination/name)\n\n\ndef show_downloads(out):\n    from IPython.display import FileLink, display\n    out = Path(out)\n    for relative in ("sanity_report.json", "results.zip", "sanity_traces.zip", "archive_status.json", "setup_status.json",\n                     "run_failure.json", "run/predictions_apc_vis.jsonl", "run/worker.log"):\n        if (out/relative).is_file():\n            display(FileLink(str(out/relative)))\n\n\ndef launch(config, python, scripts, env, retry_failed=None):\n    out = Path(config["output"])\n    folder = out/"run"\n    folder.mkdir(exist_ok=True)\n    state_path = folder/"state.json"\n    state_path.unlink(missing_ok=True)\n    atomic_json(out/"sanity_report.json", dict(technical_status="RUNNING", note="Wait for all selected APC samples."))\n    command = [str(python), "-u", str(Path(scripts)/"apc_worker.py"), "--manifest", str(out/"manifest.json")]\n    if retry_failed is True:\n        command.append("--retry-failed")\n    elif retry_failed is False:\n        command.append("--no-retry-failed")\n    started = time.monotonic()\n    failure = None\n    with (folder/"worker.log").open("a", encoding="utf-8") as stream:\n        child = subprocess.Popen(command, env=env, cwd=config["repo"], stdout=stream, stderr=subprocess.STDOUT)\n        try:\n            while child.poll() is None:\n                try:\n                    state = json.loads(state_path.read_text()) if state_path.exists() else {}\n                except (OSError, ValueError):\n                    state = {}\n                elapsed = time.monotonic()-started\n                print("APC", state.get("pipeline_stage", state.get("stage", "starting")),\n                      state.get("paper_task", ""), state.get("sample_id", ""),\n                      f"{state.get(\'processed\', 0)}/{state.get(\'total\', \'?\')}",\n                      "errors=", state.get("errors", 0), "invalid=", state.get("invalid", 0),\n                      "elapsed min=", round(elapsed/60, 1), "free GiB=", state.get("disk_free_gib"), flush=True)\n                sample_start = state.get("sample_started_unix")\n                if sample_start and time.time()-sample_start > config["sample_timeout_seconds"]+120:\n                    raise TimeoutError("Parent watchdog: sample stalled beyond timeout + 120s")\n                if not state.get("new_processed") and not sample_start and elapsed > config["load_timeout_seconds"]+120:\n                    raise TimeoutError("Parent watchdog: APC model initialization stalled")\n                if elapsed > config["run_timeout_seconds"]+120:\n                    raise TimeoutError("Parent watchdog: run time budget exceeded")\n                time.sleep(10)\n            if child.returncode:\n                with (folder/"worker.log").open("rb") as log:\n                    log.seek(max(0, (folder/"worker.log").stat().st_size-16000))\n                    print(log.read().decode("utf-8", errors="replace"))\n                raise RuntimeError(f"APC worker failed (exit {child.returncode}); inspect run_failure.json / worker.log")\n        except BaseException as exc:\n            failure = exc\n        finally:\n            terminate_worker(child)\n    # No ZIP operation races a live writer. Metadata remains downloadable if ZIP fails.\n    if failure:\n        try:\n            rows = [json.loads(line) for line in (out/"samples.jsonl").read_text().splitlines()]\n            records = read_records(folder/"predictions_apc_vis.jsonl", repair=True)\n            report_path = out/"sanity_report.json"\n            report = json.loads(report_path.read_text())\n            if report.get("technical_status") != "FAIL":\n                report = make_report(rows, records, str(failure))\n            report["supervisor_error"] = str(failure)\n            atomic_json(report_path, report)\n        except Exception as report_error:\n            print("Could not update report:", repr(report_error), flush=True)\n    try:\n        collect_reports(config["work"], out)\n    except OSError as export_error:\n        print("Report collection:", repr(export_error), flush=True)\n    export_safely(out, config, debug=True)\n    show_downloads(out)\n    if failure:\n        raise failure\n    report = json.loads((out/"sanity_report.json").read_text())\n    if report["technical_status"] != "PASS":\n        raise RuntimeError("Sanity report did not pass")\n    print("APC SANITY PASS:", report["valid_predictions"], "/", report["expected_rows"], flush=True)\n    return report\n',encoding='utf-8')
(SCRIPTS/'apc_full_reference.py').write_text('"""Normalize VLM entity labels and resolve a real scene viewer without camera fallback."""\n\nimport ast\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport unicodedata\n\n\nclass ReferenceViewerError(ValueError):\n    pass\n\n\ndef clean_label(value):\n    value = unicodedata.normalize("NFKC", str(value)).strip()\n    value = value.translate(str.maketrans({"‘":"\'", "’":"\'", "“":\'"\', "”":\'"\'}))\n    while len(value) >= 2 and value[0] == value[-1] and value[0] in "\'\\"`":\n        value = value[1:-1].strip()\n    return re.sub(r"\\s+", " ", value).casefold()\n\n\ndef canonical(value):\n    return re.sub(r"^(?:the|a|an)\\s+", "", clean_label(value))\n\n\ndef viewer_key(value):\n    # Only matching keys change. Keep the scene label used to index the geometry.\n    value = canonical(value).replace(\'"\', \'\').replace(\'`\', \'\')\n    value = re.sub(r"(?<!\\w)\'|\'(?!\\w)", "", value)\n    value = re.sub(r"\\bwomen\\b", "woman", value)\n    return re.sub(r"\\s+", " ", value).strip()\n\n\ndef parse_objects(text):\n    try:\n        values = ast.literal_eval(text)\n    except (SyntaxError, ValueError):\n        values = text.strip().removeprefix("[").removesuffix("]").split(",")\n    if not isinstance(values, (list, tuple)) or any(not isinstance(value, str) for value in values):\n        raise ValueError("VLM object list must contain strings")\n    result, seen = [], set()\n    for value in values:\n        value = clean_label(value)\n        if value and canonical(value) not in seen:\n            result.append(value)\n            seen.add(canonical(value))\n    if not result:\n        raise ValueError("VLM returned an empty object list")\n    return result\n\n\ndef explicit_subject(question):\n    # These are question templates, not answer labels. No ground truth is used.\n    patterns = (\n        r"\\b(?:if i|if you|you are|i am) (?:stand|standing) at (.+?) position\\b",\n        r"\\bfrom (.+?)(?:\'s|’s) (?:perspective|viewpoint)\\b",\n        r"\\bwhich object is (.+?) facing towards\\b",\n    )\n    for pattern in patterns:\n        match = re.search(pattern, question, flags=re.I)\n        if match:\n            return re.sub(r"(?:\'s|’s|[\'’])$", "", match.group(1).strip())\n    return None\n\n\ndef resolve_viewer(response, question, scene, trace=None):\n    matches = re.findall(r"\\+\\+(.*?)\\+\\+", response, flags=re.S)\n    candidate = matches[-1] if matches else response.strip()\n    candidate = re.sub(r"^\\s*\\[?perspective\\]?\\s*:?\\s*", "", candidate, flags=re.I)\n    candidate = clean_label(candidate)\n    subject = explicit_subject(question)\n    def matching(label, include_camera=False):\n        key = viewer_key(label) if label is not None else ""\n        return [obj for obj in scene if key and (include_camera or viewer_key(obj) != "camera")\n                and viewer_key(obj) == key]\n\n    expected_keys, proposed_keys = matching(subject), matching(candidate)\n    expected = expected_keys[0] if len(expected_keys) == 1 else None\n    proposed = proposed_keys[0] if len(proposed_keys) == 1 else None\n    # An explicit, uniquely identified benchmark subject constrains the viewpoint.\n    if expected is not None:\n        viewer, strategy = expected, "question_subject" if proposed != expected else "vlm_and_question_agree"\n    elif len(expected_keys) > 1:\n        viewer, strategy = None, "ambiguous_question_subject"\n    elif proposed is not None:\n        viewer, strategy = proposed, "unique_vlm_scene_label"\n    elif (subject is None or canonical(subject) == "camera") and canonical(candidate) == "camera" and re.search(r"\\bcamera(?:\'s)? (?:perspective|viewpoint)\\b", question, re.I):\n        camera_keys = matching("camera", include_camera=True)\n        viewer, strategy = (camera_keys[0] if len(camera_keys) == 1 else None), "explicit_camera_question"\n    else:\n        viewer, strategy = None, "unresolved"\n    audit = dict(raw_response=response, candidate=candidate, question_subject=subject,\n                 available_objects=list(scene), subject_matches=expected_keys, candidate_matches=proposed_keys,\n                 resolved_viewer=viewer, strategy=strategy)\n    folder = trace or os.environ.get("APC_TRACE_DIR")\n    if folder:\n        (Path(folder)/"reference_viewer.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")\n    if viewer is None:\n        raise ReferenceViewerError("No unique reference viewer in the detected scene; inspect reference_viewer.json")\n    return viewer\n\n\ndef patch_reference_source(source):\n    if "from apc_full_reference import resolve_viewer" in source:\n        return source\n    tree = ast.parse(source)\n    methods = {node.name:node for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)}\n    objects = methods["get_objects_of_interest"]\n    assignments = [node for node in ast.walk(objects) if isinstance(node, ast.Assign)\n                   and any(isinstance(target, ast.Name) and target.id == "objs_of_interest" for target in node.targets)]\n    start, end = assignments[0].lineno-1, assignments[-1].end_lineno\n    lines = source.splitlines(keepends=True)\n    lines[start:end] = ["        from apc_full_reference import parse_objects\\n",\n                        "        objs_of_interest = parse_objects(match_objs)\\n"]\n    source = "".join(lines)\n    tree = ast.parse(source)\n    method = next(node for node in ast.walk(tree) if isinstance(node, ast.FunctionDef) and node.name == "get_reference_viewer")\n    first = next(node for node in method.body if isinstance(node, ast.Assign)\n                 and any(isinstance(target, ast.Name) and target.id == "pattern_ref_viewer" for target in node.targets))\n    logger = next(node for node in method.body if isinstance(node, ast.Expr) and isinstance(node.value, ast.Call)\n                  and getattr(node.value.func, "attr", "") == "info")\n    lines = source.splitlines(keepends=True)\n    lines[first.lineno-1:logger.end_lineno] = [\n        "        from apc_full_reference import resolve_viewer\\n",\n        "        ref_viewer = resolve_viewer(response_ref_viewer, prompt, abstract_scene_dict_camera)\\n",\n        "        self.logger.info(f\'[Perspective Change] Reference viewer: {ref_viewer}\')\\n",\n    ]\n    source = "".join(lines)\n    ast.parse(source)\n    return source\n\n\nif __name__ == "__main__":\n    import hashlib\n    import sys\n    target = Path(sys.argv[1])/"apc/apc_pipeline.py"\n    before = target.read_text(encoding="utf-8")\n    after = patch_reference_source(before)\n    target.write_text(after, encoding="utf-8")\n    Path(sys.argv[2]).write_text(json.dumps(dict(\n        policy="unique_scene_key_with_quote_and_person_number_normalization_v2",\n        before_sha=hashlib.sha256(before.encode()).hexdigest(),\n        after_sha=hashlib.sha256(after.encode()).hexdigest()), indent=2), encoding="utf-8")\n',encoding='utf-8')
(SCRIPTS/'apc_full_checkpoint.py').write_text('"""Portable full-run checkpoints, pinned resume assets and full-denominator reports."""\n\nfrom collections import defaultdict\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport zipfile\n\nfrom apc_kaggle_guardrails import (\n    OPERATIONAL_KEYS, atomic_json, export_safely, make_report, read_records,\n    select_samples as select_base, successful,\n)\n\n\ndef select_samples(rows, config):\n    rows = select_base(rows, config)\n    if config["run_kind"] != "full":\n        return rows\n    tasks = config["paper_tasks"]\n    grouped = {task:{} for task in tasks}\n    for row in rows:\n        grouped[row["paper_task"]].setdefault(row["circular_group"], []).append(row)\n    lists = [list(grouped[task].values()) for task in tasks]\n    # Exercise both part-1 task paths early; preserve every complete circular pair.\n    return [row for index in range(max(map(len, lists))) for groups in lists if index < len(groups)\n            for row in groups[index]]\n\n\ndef prepare_resume(config, work):\n    source = config.get("resume_input")\n    if not source:\n        return\n    work, source = Path(work), Path(source)\n    if source.is_file():\n        destination = work/"resume"/config["experiment"]\n        destination.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(source) as archive:\n            for name in ("manifest.json", "run/predictions_apc_vis.jsonl"):\n                info = archive.getinfo(name)\n                if info.file_size > 128*2**20:\n                    raise ValueError("Resume entry exceeds 128 MiB: " + name)\n                target = destination/name\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(archive.read(info))\n        source = destination\n    manifest = json.loads((source/"manifest.json").read_text(encoding="utf-8"))\n    old_config = {k:v for k,v in manifest["config"].items() if k not in OPERATIONAL_KEYS}\n    current = {k:v for k,v in config.items() if k not in OPERATIONAL_KEYS}\n    if old_config != current:\n        raise RuntimeError("Resume protocol differs: use the same v5 part/model/code configuration.")\n    if not (source/"run/predictions_apc_vis.jsonl").is_file():\n        raise ValueError("Resume folder is missing run/predictions_apc_vis.jsonl")\n    # Restore exact revisions BEFORE native install and auxiliary-model downloads.\n    atomic_json(work/"source_lock.json", manifest["source_lock"])\n    assets = manifest["assets"]\n    root = Path(config["asset_input"] or config["asset_root"])\n    if config["asset_input"]:\n        existing = json.loads((root/"assets_manifest.json").read_text())\n        if {k:v for k,v in existing.items() if k != "root"} != {k:v for k,v in assets.items() if k != "root"}:\n            raise RuntimeError("Mounted assets differ from the checkpoint\'s pinned assets")\n    else:\n        root.mkdir(parents=True, exist_ok=True)\n        atomic_json(root/"assets_manifest.json", assets)\n    config["resume_input"] = str(source.resolve())\n    print("Resume loaded; native/model revisions pinned to previous manifest:", source, flush=True)\n\n\ndef ensure_checkout(path, url, sha):\n    """Fetch an immutable commit in a generated runtime copy, preserving dirty caches."""\n    path = Path(path)\n    if not re.fullmatch(r"[0-9a-f]{40}|[0-9a-f]{64}", sha):\n        raise ValueError("Invalid pinned git SHA")\n    def git(*args, capture=False):\n        command = ["git", "-C", str(path), *args]\n        return subprocess.check_output(command, text=True).strip() if capture else subprocess.run(command, check=True)\n    path.mkdir(parents=True, exist_ok=True)\n    if not (path/".git").exists():\n        git("init", "--quiet")\n        git("remote", "add", "origin", url)\n    else:\n        current = git("rev-parse", "HEAD", capture=True)\n        if current == sha:\n            return\n        if git("status", "--porcelain", capture=True):\n            raise RuntimeError("Cached native source differs from resume and has patches; resume in a clean Kaggle session.")\n    git("fetch", "--depth", "1", "origin", sha)\n    git("checkout", "--detach", sha)\n    if git("rev-parse", "HEAD", capture=True) != sha:\n        raise RuntimeError("Pinned native checkout failed")\n\n\ndef compact_record(record):\n    result = {k:v for k,v in record.items() if k not in ("vlm_calls", "stages")}\n    for call in record.get("vlm_calls", []):\n        for key in ("reference_viewer_strategy", "resolved_viewer"):\n            if key in call:\n                result[key] = call[key]\n    return result\n\n\ndef write_full_report(out, *, pause=False, failure=None):\n    out = Path(out)\n    manifest = json.loads((out/"manifest.json").read_text())\n    rows = [json.loads(line) for line in (out/"samples.jsonl").read_text().splitlines()]\n    records = read_records(out/"run/predictions_apc_vis.jsonl", repair=True)\n    records = {sid:compact_record(record) for sid,record in records.items()}\n    report = make_report(rows, records, str(failure) if failure else None)\n    report.update(completion_status="PAUSED" if pause else "ERROR" if failure else\n                  "COMPLETE" if len(records) == len(rows) else "PARTIAL",\n                  attempted_percent=100*len(records)/len(rows),\n                  valid_percent=100*report["valid_predictions"]/len(rows),\n                  all_rows_attempted=len(records)==len(rows),\n                  complete=report["valid_predictions"]==len(rows) and failure is None,\n                  circular_groups=len({r["circular_group"] for r in rows}),\n                  accuracy_note="All selected rows/groups stay in denominator; missing, error and invalid count as wrong.",\n                  protocol=manifest["protocol"], fingerprint=manifest["fingerprint"])\n    if pause:\n        report["technical_status"] = "PARTIAL"\n        report["stop_reason"] = "session_time_budget; resume from checkpoint.zip"\n    report["quality_warnings"] = {sid:r["quality_warnings"] for sid,r in records.items() if r.get("quality_warnings")}\n    report["reference_viewer_repairs"] = sum(r.get("reference_viewer_strategy") == "question_subject" for r in records.values())\n    report["peak_vram_gib"] = {\n        str(gpu):max((r.get("peak_vram_gib", {}).get(str(gpu), 0) for r in records.values() if r.get("peak_vram_gib")), default=0)\n        for gpu in (0, 1)}\n    grouped = defaultdict(list)\n    for row in rows:\n        grouped[row["circular_group"]].append(row)\n    def hit(row):\n        prediction = records.get(str(row["sample_id"]), {})\n        return successful(prediction) and prediction.get("prediction") == row["answer"]\n    for task, metric in report["by_task"].items():\n        selected = [row for row in rows if row["paper_task"] == task]\n        circular = [group for group in grouped.values() if group[0]["paper_task"] == task]\n        metric.update(row_accuracy=sum(hit(row) for row in selected)/len(selected),\n                      circular_groups=len(circular),\n                      circular_accuracy=sum(all(hit(row) for row in group) for group in circular)/len(circular))\n    for filename in ("full_report.json", "summary.json"):\n        atomic_json(out/filename, report)\n    # Compact merged rows support analysis without the per-call prompts in debug traces.\n    with (out/"apc_vis_predictions.jsonl").open("w", encoding="utf-8") as stream:\n        for row in rows:\n            prediction = records.get(str(row["sample_id"]), dict(status="missing", prediction=None))\n            merged = dict(row, **{k:v for k,v in prediction.items() if k not in ("vlm_calls", "stages")})\n            merged["correct"] = bool(hit(row))\n            stream.write(json.dumps(merged, ensure_ascii=False)+"\\n")\n    return report\n\n\ndef create_checkpoint(out, config):\n    out = Path(out)\n    target = out/"checkpoint.zip"\n    temporary = target.with_suffix(".zip.tmp")\n    records = read_records(out/"run/predictions_apc_vis.jsonl", repair=True)\n    portable = []\n    for record in records.values():\n        value = compact_record(record)\n        if value.get("trace_path"):\n            value["previous_trace_path"] = value["trace_path"]\n        value["trace_path"] = None  # Debug files are deliberately outside checkpoint.zip.\n        portable.append(value)\n    lines = "".join(json.dumps(record, ensure_ascii=False)+"\\n" for record in portable).encode("utf-8")\n    manifest = (out/"manifest.json").read_bytes()\n    required = len(lines)+len(manifest)+2**20\n    if required > 128*2**20 or shutil.disk_usage(out).free < config["disk_min_free_gib"]*2**30+required:\n        target.unlink(missing_ok=True)\n        return dict(status="SKIPPED", reason="checkpoint size/free disk budget")\n    try:\n        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:\n            archive.writestr("manifest.json", manifest)\n            archive.writestr("run/predictions_apc_vis.jsonl", lines)\n        temporary.replace(target)\n    except BaseException:\n        temporary.unlink(missing_ok=True)\n        target.unlink(missing_ok=True)\n        raise\n    return dict(status="OK", path=str(target), rows=len(records), bytes=target.stat().st_size)\n\n\ndef export_full(out, config):\n    out = Path(out)\n    try:\n        checkpoint = create_checkpoint(out, config) if (out/"manifest.json").exists() else dict(status="NOT_READY")\n    except Exception as exc:\n        checkpoint = dict(status="FAILED", message=str(exc))\n    try:\n        atomic_json(out/"checkpoint_status.json", checkpoint)\n    except OSError:\n        pass\n    # The retained debug archive is small; primary results never contain this tree.\n    (out/"traces.zip").unlink(missing_ok=True)\n    status = export_safely(out, config, debug=True)\n    traces = out/"sanity_traces.zip"\n    if status.get("traces", {}).get("status") == "OK":\n        try:\n            traces.replace(out/"traces.zip")\n            status["traces"]["path"] = str(out/"traces.zip")\n        except OSError as exc:\n            status["traces"] = dict(status="FAILED", message=str(exc))\n    status["checkpoint"] = checkpoint\n    try:\n        atomic_json(out/"archive_status.json", status)\n    except OSError:\n        pass\n    print("Checkpoint:", json.dumps(checkpoint), flush=True)\n    return status\n',encoding='utf-8')

sys.path.insert(0,str(SCRIPTS))
for module in ('apc_launcher','apc_v4_launcher','apc_full_checkpoint','apc_kaggle_guardrails'):
    sys.modules.pop(module,None)
from apc_kaggle_guardrails import setup_stage,run_logged
from apc_full_checkpoint import prepare_resume,export_full
from apc_launcher import show_downloads
print('Runtime v6 written: viewer matching and sample-level error handling.')
import base64, hashlib
from pathlib import Path
_payload = base64.b64decode('...')
assert hashlib.sha256(_payload).hexdigest() == '9653b3685d9e4b96989e65f13d1f806c1604bd7fd17a58e44a424a64bc28dfbd'
_seed_path = Path('/kaggle/temp/apc_completion_seed.zip'); _seed_path.parent.mkdir(parents=True, exist_ok=True)
_seed_path.write_bytes(_payload)
del _payload
import base64, hashlib
from pathlib import Path
_payload = base64.b64decode('...')
assert hashlib.sha256(_payload).hexdigest() == '84e9641b78bc25992f28c6423e20466fd041a7959e4c369e3bc30e86bef2e48a'
_seed_path = Path('/kaggle/temp/apc_completion_reference.zip'); _seed_path.parent.mkdir(parents=True, exist_ok=True)
_seed_path.write_bytes(_payload)
del _payload

(SCRIPTS/'completion_helpers.py').write_text('"""Bookkeeping for a frozen benchmark continuation and independent diagnostic replay."""\nimport collections, hashlib, json, pathlib, shutil, subprocess, zipfile\nfrom apc_kaggle_guardrails import atomic_json, protocol_fingerprint, read_records\n\ndef jsonl(path):\n    return [json.loads(x) for x in pathlib.Path(path).read_text(encoding=\'utf8\').splitlines() if x.strip()]\n\ndef load_reference(path):\n    with zipfile.ZipFile(path) as z:\n        return {name:json.loads(z.read(name)) if name.endswith(\'.json\') else\n                [json.loads(x) for x in z.read(name).decode(\'utf8\').splitlines() if x.strip()]\n                for name in z.namelist()}\n\ndef outcome(row):\n    if row.get(\'status\')!=\'ok\':return row.get(\'status\',\'missing\')\n    return \'correct\' if row.get(\'prediction\')==row[\'answer\'] else \'wrong\'\n\ndef archive_files(folder,target,names):\n    folder,target=pathlib.Path(folder),pathlib.Path(target)\n    names=[pathlib.Path(n) for n in names if (folder/n).is_file()]\n    needed=sum((folder/n).stat().st_size for n in names)+256*2**20\n    if shutil.disk_usage(folder).free < needed+int(1.5*2**30):\n        print(\'ZIP deferred: keep raw Output files; insufficient archive reserve:\',target)\n        return False\n    temp=target.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(temp,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for n in names:z.write(folder/n,n.as_posix())\n    temp.replace(target)\n    return True\n\ndef merge_benchmark(part1,reference,out):\n    part1,out=pathlib.Path(part1),pathlib.Path(out);out.mkdir(parents=True,exist_ok=True)\n    ref=load_reference(reference)\n    rows=jsonl(part1/\'apc_vis_predictions.jsonl\')+ref[\'part2_predictions.jsonl\']\n    assert len(rows)==2078 and len({r[\'sample_id\'] for r in rows})==2078\n    with (out/\'predictions.jsonl\').open(\'w\',encoding=\'utf8\') as f:\n        for r in rows:f.write(json.dumps(r,ensure_ascii=False)+\'\\n\')\n    report={}\n    for task in [\'left_right\',\'visibility\',\'facing\']:\n        rr=[r for r in rows if r[\'paper_task\']==task];groups=collections.defaultdict(list)\n        for r in rr:groups[r[\'circular_group\']].append(r)\n        assert all(len(g)==2 for g in groups.values())\n        count=collections.Counter(outcome(r) for r in rr)\n        report[task]=dict(expected=len(rr),attempted=len(rr)-count[\'missing\'],missing=count[\'missing\'],\n            statuses=dict(count),row_accuracy=sum(outcome(r)==\'correct\' for r in rr)/len(rr),\n            circular_accuracy=sum(all(outcome(r)==\'correct\' for r in g) for g in groups.values())/len(groups))\n    report[\'complete\']=all(r.get(\'status\')!=\'missing\' for r in rows)\n    report[\'note\']=\'Frozen part 2 + resumed part 1; replay is excluded. Errors remain in denominators.\'\n    atomic_json(out/\'summary.json\',report)\n    archive_files(out,out/\'benchmark_results.zip\',[\'predictions.jsonl\',\'summary.json\'])\n    print(\'Combined benchmark: complete=\',report[\'complete\'],flush=True)\n    return report\n\ndef prepare_replay(config,part1,reference,work,scripts,python,env,resume_input=None):\n    work,part1=pathlib.Path(work),pathlib.Path(part1)\n    ref=load_reference(reference);selection=ref[\'trace_selection.json\']\n    assert len(selection)==120 and len({r[\'sample_id\'] for r in selection})==120\n    prep=work/\'trace_prepare\';prep.mkdir(exist_ok=True)\n    trace_out=part1.parent/\'trace_replay_120\';trace_out.mkdir(exist_ok=True)\n    cfg=dict(config,experiment=\'apc_trace_replay_120_v2\',output=str(prep),resume_input=None,\n        paper_tasks=[\'left_right\',\'visibility\',\'facing\'],expected_rows=2078,expected_circular_groups=1039,\n        expected_task_counts=dict(left_right=698,visibility=688,facing=692),\n        trace_keep_per_task=40,trace_max_total_mib=2048,trace_max_sample_mib=128,\n        run_timeout_seconds=10800,retry_failed=False)\n    path=work/\'trace_prepare_config.json\';path.write_text(json.dumps(cfg),encoding=\'utf8\')\n    subprocess.run([str(python),str(pathlib.Path(scripts)/\'apc_prepare_data.py\'),str(path)],env=env,cwd=config[\'repo\'],check=True)\n    manifest=json.loads((prep/\'manifest.json\').read_text())\n    available={r[\'sample_id\']:r for r in jsonl(prep/\'samples.jsonl\')}\n    rows=[available[s[\'sample_id\']] for s in selection]\n    for r,s in zip(rows,selection):assert r[\'image_sha256\']==s[\'image_sha256\'] and r[\'paper_task\']==s[\'paper_task\']\n    cfg.update(output=str(trace_out),expected_rows=120,expected_circular_groups=120,\n               expected_task_counts=dict(left_right=40,visibility=40,facing=40))\n    manifest.update(config=cfg,selected_rows=120,circular_groups=120,tasks=cfg[\'expected_task_counts\'],\n        protocol=\'Diagnostic replay selected from previous answers: 28 wrong + 12 correct per task; NOT benchmark accuracy\')\n    manifest[\'fingerprint\']=protocol_fingerprint(manifest,rows)\n    if resume_input and not (trace_out/\'run/predictions_apc_vis.jsonl\').exists():\n        with zipfile.ZipFile(resume_input) as z:\n            previous=json.loads(z.read(\'manifest.json\'))\n            if previous[\'fingerprint\']!=manifest[\'fingerprint\']:raise ValueError(\'Trace resume fingerprint mismatch\')\n            infos=z.infolist()\n            if sum(i.file_size for i in infos)>3*2**30:raise ValueError(\'Unexpectedly large trace archive\')\n            if shutil.disk_usage(trace_out).free < sum(i.file_size for i in infos)+int(1.5*2**30):raise OSError(\'Not enough disk for trace resume\')\n            for i in infos:\n                pp=pathlib.PurePosixPath(i.filename)\n                if pp.is_absolute() or \'..\' in pp.parts or \'\\\\\' in i.filename:raise ValueError(\'Unsafe ZIP member\')\n                if i.is_dir():continue\n                target=trace_out.joinpath(*pp.parts)\n                target.resolve().relative_to(trace_out.resolve())\n                target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(i))\n    previous=trace_out/\'manifest.json\'\n    if previous.exists() and json.loads(previous.read_text())[\'fingerprint\']!=manifest[\'fingerprint\']:\n        raise ValueError(\'Existing replay output has another fingerprint\')\n    atomic_json(previous,manifest)\n    atomic_json(trace_out/\'config.json\',cfg)\n    atomic_json(trace_out/\'selection.json\',selection)\n    (trace_out/\'samples.jsonl\').write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in rows),encoding=\'utf8\')\n    # A portable trace resume must contain the trace, not only the answer.\n    journal=trace_out/\'run/predictions_apc_vis.jsonl\'\n    saved=read_records(journal,repair=True)\n    valid=[];dropped=[]\n    for r in saved.values():\n        path=r.get(\'trace_path\')\n        target=(trace_out/\'run\'/path) if path else None\n        if target is not None:\n            target.resolve().relative_to((trace_out/\'run/trace\').resolve())\n        if target is not None and target.is_dir() and any(target.iterdir()):valid.append(r)\n        else:dropped.append(r)\n    if dropped:\n        with (trace_out/\'previous_missing_trace_attempts.jsonl\').open(\'a\',encoding=\'utf8\') as f:\n            for r in dropped:f.write(json.dumps(r)+\'\\n\')\n        tmp=journal.with_suffix(\'.tmp\');tmp.write_text(\'\'.join(json.dumps(r)+\'\\n\' for r in valid),encoding=\'utf8\');tmp.replace(journal)\n    print(\'Replay: selected 120 distinct task/image cases, retained prior trace records:\',len(valid),flush=True)\n    return cfg\n\ndef finish_replay(out,reference):\n    out=pathlib.Path(out);ref=load_reference(reference)\n    selection=ref[\'trace_selection.json\'];saved=read_records(out/\'run/predictions_apc_vis.jsonl\',repair=True)\n    sample_index={r[\'sample_id\']:r for r in jsonl(out/\'samples.jsonl\')}\n    result=[]\n    for s in selection:\n        r=saved.get(s[\'sample_id\'],{});row=sample_index[s[\'sample_id\']]\n        path=out/\'run\'/r[\'trace_path\'] if r.get(\'trace_path\') else None\n        exists=path is not None and path.is_dir() and any(path.iterdir())\n        result.append(dict(s,replay_status=r.get(\'status\',\'missing\'),replay_prediction=r.get(\'prediction\'),\n            replay_correct=r.get(\'status\')==\'ok\' and r.get(\'prediction\')==row[\'answer\'],\n            prediction_changed=bool(r) and r.get(\'prediction\')!=s[\'prior_prediction\'],\n            trace_saved=exists,trace_path=r.get(\'trace_path\'),\n            failed_stage=r.get(\'failed_stage\'),message=r.get(\'message\')))\n    by_task={}\n    for task in [\'left_right\',\'visibility\',\'facing\']:\n        rr=[r for r in result if r[\'paper_task\']==task]\n        by_task[task]=dict(selected=len(rr),trace_saved=sum(r[\'trace_saved\'] for r in rr),\n            replay_correct=sum(r[\'replay_correct\'] for r in rr),\n            replay_wrong=sum(r[\'replay_status\']==\'ok\' and not r[\'replay_correct\'] for r in rr),\n            invalid_or_error=sum(r[\'replay_status\'] in [\'invalid\',\'error\'] for r in rr),\n            missing=sum(r[\'replay_status\']==\'missing\' for r in rr))\n    report=dict(complete=all(r[\'trace_saved\'] for r in result),by_task=by_task,cases=result,\n        note=\'28 wrong + 12 correct selected using PRIOR output per task. Replayed labels may change. No benchmark predictions overwritten.\')\n    atomic_json(out/\'replay_comparison.json\',report)\n    # One selected row per image is not a full circular pair. Do not present\n    # the generic worker\'s single-row group score as benchmark CircularEval.\n    for filename in [\'full_report.json\',\'summary.json\']:\n        path=out/filename\n        if path.is_file():\n            value=json.loads(path.read_text())\n            value[\'protocol\']=\'Diagnostic replay; purposefully selected previous failures and controls\'\n            value[\'accuracy_note\']=\'Selected sample only, excluded from the 2078-row benchmark. CircularEval not applicable.\'\n            value[\'circular_accuracy\']=None\n            for metric in value.get(\'by_task\',{}).values():metric[\'circular_accuracy\']=None\n            atomic_json(path,value)\n    result_names=[p.relative_to(out) for p in out.rglob(\'*\') if p.is_file() and\n        \'trace\' not in p.relative_to(out).parts and p.suffix in [\'.json\',\'.jsonl\',\'.txt\',\'.log\']]\n    archive_files(out,out/\'results.zip\',result_names)\n    names=[p.relative_to(out) for p in out.rglob(\'*\') if p.is_file() and\n        (p.suffix in [\'.json\',\'.jsonl\'] or \'trace\' in p.relative_to(out).parts)]\n    archive_files(out,out/\'trace_resume.zip\',names)\n    print(json.dumps(dict(complete=report[\'complete\'],by_task=by_task),indent=2),flush=True)\n    return report\n',encoding='utf8')


## Resume cấu hình đã chạy thành công
Giữ model, prompt, precision, parser, viewer policy và seed. Chỉ đổi đường dẫn runtime/output và ngân sách lưu trace.

In [ ]:
# Optional: point this to a NEWER part-1 checkpoint if continuing another session.
APC_RESUME_CHECKPOINT = '/kaggle/temp/apc_completion_seed.zip'
# Trace resume ZIP from this notebook; None starts the selected replay from scratch.
TRACE_RESUME_INPUT = None
RUN_TRACE_REPLAY = True
CFG = {'model_id': 'Qwen/Qwen2.5-VL-7B-Instruct', 'model_revision': 'cc594898137f460bfe9f0759e9844b3ce807cfb5', 'dataset_id': 'ccvl/3DSRBench', 'data_revision': '5c90e422b68d0484d50a3ff06b3096ea7d288026', 'data_file': '3dsrbench_v1_vlmevalkit_circular.tsv', 'dataset_input': None, 'asset_input': None, 'asset_root': '/kaggle/temp/apc_7b_assets', 'experiment': 'apc_part1_complete_20261112', 'scope': 'paper_tasks_visibility', 'modes': ['apc_vis'], 'seed': 42, 'dtype': 'float16', 'min_pixels': None, 'max_pixels': None, 'intermediate_max_new_tokens': 1024, 'final_max_new_tokens': 512, 'max_input_tokens': 32768, 'detection_box_threshold': 0.05, 'detection_text_threshold': 0.05, 'limit': None, 'sample_timeout_seconds': 900, 'load_timeout_seconds': 1800, 'retry_failed': False, 'parse_policy': 'strict_letter_or_unique_exact_choice_v1', 'source_snapshot_sha': '9059ed359baf89b49675ed3e0407f843b3ff28de78e5cae1c870adf79f44a339', 'runtime_sha': '178bd62f6a753f5731d940d9f876596ceeb239b640d9d789f53aa896d4f27283', 'sanity': False, 'vlm_activation_reserve_gib': [3.0, 2.0], 'vision_reserve_gib': 6.0, 'vision_offload': True, 'run_kind': 'full', 'paper_tasks': ['left_right', 'visibility'], 'resume_input': '/kaggle/temp/apc_completion_seed.zip', 'disk_min_free_gib': 1.5, 'trace_max_total_mib': 128, 'trace_max_sample_mib': 128, 'trace_keep_per_task': 0, 'archive_max_mib': 64, 'max_consecutive_errors': 3, 'max_total_errors': 25, 'run_timeout_seconds': 18000, 'expected_rows': 1386, 'expected_circular_groups': 693, 'expected_task_counts': {'left_right': 698, 'visibility': 688}, 'reference_viewer_policy': 'unique_scene_key_with_quote_and_person_number_normalization_v2', 'work': '/kaggle/temp/apc_completion_runtime', 'repo': '/kaggle/temp/apc_completion_runtime/repo_9059ed359baf', 'output': '/kaggle/working/apc_completion/part1'}
CFG.update(work=str(WORK), repo=str(REPO), resume_input=APC_RESUME_CHECKPOINT)
OUT=Path(CFG['output']); OUT.mkdir(parents=True,exist_ok=True)
CONFIG=WORK/'completion_config.json'
with setup_stage('resume_configuration',OUT,CFG):
    prepare_resume(CFG,WORK)
CONFIG.write_text(json.dumps(CFG,indent=2),encoding='utf8')
(OUT/'config.json').write_text(json.dumps(CFG,indent=2),encoding='utf8')
print('Part 1: retain 1011 rows; original pending count 375. No retries of completed rows.')


## Python, CUDA và chia Qwen trên hai GPU

Venv Python 3.10 + Torch 2.4.1/cu124; giữ bản sửa device map v3 và probe dispatch thật.
Các model vision lần lượt chạy trên GPU 1, về RAM khi chưa dùng. T4 ×2 cùng phục vụ một APC pipeline.

In [ ]:
with setup_stage('python_cuda_dispatch', OUT, CFG):
    import csv
    smi = shutil.which('nvidia-smi')
    if not smi:
        smi = next((p for p in ('/usr/bin/nvidia-smi', '/usr/local/nvidia/bin/nvidia-smi') if Path(p).is_file()), None)
    if not smi:
        raise RuntimeError('Phiên Kaggle chưa có nvidia-smi. Chọn Settings > Accelerator > GPU T4 x2, rồi khởi động lại session. Cài package không thể cấp GPU cho phiên CPU.')
    gpu_query = subprocess.run([smi, '--query-gpu=uuid,name,driver_version,memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True)
    print(gpu_query.stdout)
    if gpu_query.returncode != 0:
        print(gpu_query.stderr)
        raise RuntimeError('NVIDIA driver chưa truy cập được. Chọn accelerator 2 GPU và khởi động lại session Kaggle.')
    gpu_rows = [row for row in csv.reader(gpu_query.stdout.splitlines()) if len(row) == 4]
    if len(gpu_rows) < 2:
        raise RuntimeError(f'Phiên này chỉ có {len(gpu_rows)} GPU. Full benchmark cần hai GPU; khởi động lại session sau khi chọn accelerator.')
    GPU_IDS = [row[0].strip() for row in gpu_rows[:2]]
    print('GPU UUID được dùng:', GPU_IDS)
    minimum_env_space = (0.25 if (WORK / 'venv_py310_uv/pyvenv.cfg').is_file() else 8) * 2 ** 30
    if shutil.disk_usage(WORK).free < minimum_env_space:
        raise RuntimeError('Không đủ working cho cài môi trường (8 GiB khi cài mới, 0.25 GiB khi tiếp tục). Dùng model_input hoặc session mới nếu cần.')
    import io, platform, tarfile, urllib.request
    assert platform.system() == 'Linux' and platform.machine() in ('x86_64', 'AMD64'), 'Cell này dành cho Kaggle Linux x64.'
    UV_VERSION = '0.9.5'
    UV_DIR = WORK / 'tools' / ('uv-' + UV_VERSION)
    UV = UV_DIR / 'uv'
    if not UV.exists():
        asset = 'uv-x86_64-unknown-linux-gnu.tar.gz'
        base_url = f'https://github.com/astral-sh/uv/releases/download/{UV_VERSION}/'
        with urllib.request.urlopen(base_url + asset, timeout=120) as response:
            raw = response.read()
        with urllib.request.urlopen(base_url + asset + '.sha256', timeout=120) as response:
            expected = response.read().decode().split()[0]
        assert hashlib.sha256(raw).hexdigest() == expected, 'uv checksum mismatch'
        UV_DIR.mkdir(parents=True, exist_ok=True)
        with tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz') as archive:
            members = [m for m in archive.getmembers() if Path(m.name).name == 'uv' and m.isfile()]
            assert len(members) == 1
            with archive.extractfile(members[0]) as binary:
                UV.write_bytes(binary.read())
        UV.chmod(493)
    UV.chmod(UV.stat().st_mode | 73)
    BOOT_ENV = os.environ.copy()
    for key in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV'):
        BOOT_ENV.pop(key, None)
    BOOT_ENV['PYTHONNOUSERSITE'] = '1'
    BOOT_ENV['UV_PYTHON_INSTALL_DIR'] = str(WORK / 'python_runtimes')
    BOOT_ENV['UV_CACHE_DIR'] = str(WORK / 'uv_cache')
    BOOT_ENV['UV_NO_PROGRESS'] = '1'
    BOOT_ENV['UV_LINK_MODE'] = 'copy'
    BOOT_ENV.pop('UV_SYSTEM_PYTHON', None)
    BOOT_ENV['CUDA_VISIBLE_DEVICES'] = ','.join(GPU_IDS)
    driver_dirs = [p for p in ('/usr/local/nvidia/lib64', '/usr/local/nvidia/lib', '/usr/lib64-nvidia', '/usr/lib/x86_64-linux-gnu') if Path(p).is_dir()]
    inherited_dirs = [p for p in BOOT_ENV.get('LD_LIBRARY_PATH', '').split(':') if p]
    BOOT_ENV['LD_LIBRARY_PATH'] = ':'.join(dict.fromkeys(inherited_dirs + driver_dirs))
    VENV = WORK / 'venv_py310_uv'
    PY = str(VENV / 'bin/python')
    subprocess.run([str(UV), '--version'], env=BOOT_ENV, check=True)
    for binary in (WORK / 'python_runtimes').glob('cpython-3.10*/bin/python*'):
        if binary.is_file():
            binary.chmod(binary.stat().st_mode | 73)
    for binary in (VENV / 'bin').glob('*'):
        if binary.is_file():
            binary.chmod(binary.stat().st_mode | 73)
    subprocess.run([str(UV), 'python', 'install', '3.10'], env=BOOT_ENV, check=True)
    if not (VENV / 'pyvenv.cfg').is_file():
        subprocess.run([str(UV), 'venv', '--python', '3.10', '--managed-python', str(VENV)], env=BOOT_ENV, check=True)
    if not Path(PY).exists() and (VENV / 'pyvenv.cfg').is_file():
        managed = subprocess.check_output([str(UV), 'python', 'find', '--managed-python', '3.10'], env=BOOT_ENV, text=True).strip()
        executable = Path(managed).resolve(strict=True)
        executable.chmod(executable.stat().st_mode | 73)
        link = Path(PY)
        link.parent.mkdir(parents=True, exist_ok=True)
        link.unlink(missing_ok=True)
        link.symlink_to(executable)
    subprocess.run([PY, '-c', 'import sys; assert sys.version_info[:2] == (3, 10); print(sys.version)'], env=BOOT_ENV, check=True)
    subprocess.run([str(UV), 'pip', 'install', '--python', PY, '--no-cache', '--index-url', 'https://download.pytorch.org/whl/cu124', 'torch==2.4.1', 'torchvision==0.19.1'], env=BOOT_ENV, check=True)
    packages = ['numpy==1.26.4', 'transformers==4.57.1', 'tokenizers==0.22.1', 'accelerate==1.11.0', 'huggingface-hub==0.36.0', 'safetensors==0.6.2', 'Pillow==11.3.0', 'einops==0.8.1', 'timm==1.0.22', 'sentencepiece==0.2.0', 'requests', 'tqdm', 'packaging', 'psutil', 'protobuf']
    subprocess.run([str(UV), 'pip', 'install', '--python', PY, '--no-cache', *packages], env=BOOT_ENV, check=True)
    ENV = BOOT_ENV.copy()
    ENV['PYTHONNOUSERSITE'] = '1'
    ENV['HF_HOME'] = str(WORK / 'hf_cache')
    ENV['HF_HUB_DISABLE_XET'] = '1'
    ENV['OMP_NUM_THREADS'] = '2'
    ENV['OPENBLAS_NUM_THREADS'] = '2'
    probe = "\nimport ctypes, os, sys\nimport torch\nprint('Python:', sys.version.split()[0])\nprint('torch:', torch.__version__, 'CUDA wheel:', torch.version.cuda)\nprint('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))\nprint('LD_LIBRARY_PATH:', os.environ.get('LD_LIBRARY_PATH'))\ntry:\n    driver = ctypes.CDLL('libcuda.so.1')\n    rc = driver.cuInit(0)\n    print('NVIDIA driver cuInit result:', rc)\nexcept OSError as exc:\n    print('Cannot load NVIDIA driver:', exc)\navailable = torch.cuda.is_available()\ncount = torch.cuda.device_count()\nprint('CUDA available:', available, 'device count:', count)\nif not available:\n    try:\n        torch.cuda.init()\n    except Exception as exc:\n        print('CUDA initialization error:', type(exc).__name__, str(exc))\n    raise SystemExit(2)\nif count != 2:\n    print('Expected two visible GPUs.')\n    raise SystemExit(3)\n# Kiểm tra tính toán thật trên từng GPU, không chỉ liệt kê tên.\nfor i in range(count):\n    p = torch.cuda.get_device_properties(i)\n    x = torch.ones((8, 8), device=f'cuda:{i}')\n    assert (x @ x)[0, 0].item() == 8\n    torch.cuda.synchronize(i)\n    print(i, p.name, round(p.total_memory / 2**30, 2), 'GiB; compute OK')\nimport transformers\nfrom transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, AutoModel, AutoTokenizer\nimport torchvision, timm\nprint('transformers:',transformers.__version__,'torchvision:',torchvision.__version__,'timm:',timm.__version__)\n"
    probe_result = subprocess.run([PY, '-c', probe], env=ENV, capture_output=True, text=True)
    print(probe_result.stdout)
    if probe_result.stderr:
        print(probe_result.stderr)
    (WORK / 'gpu_check.txt').write_text(gpu_query.stdout + '\n' + probe_result.stdout + '\n' + probe_result.stderr, encoding='utf-8')
    if probe_result.returncode != 0:
        raise RuntimeError('GPU/import check thất bại. Xem CUDA initialization error và gpu_check.txt ở trên. Nếu nvidia-smi không có hai GPU, cần đổi accelerator và khởi động lại session; không cần cài lại package.')
    print('Môi trường sẵn sàng; GPU compute check thành công.')
    CFG['uv'] = str(UV)
    CONFIG.write_text(json.dumps(CFG, indent=2), encoding='utf-8')
    ENV['PYTHONPATH'] = str(SCRIPTS) + os.pathsep + str(REPO)
    ENV['MPLBACKEND'] = 'Agg'
    ENV['CUDA_LAUNCH_BLOCKING'] = '1' if CFG['sanity'] else '0'
    PREFLIGHT_ENV = ENV.copy()
    PREFLIGHT_ENV.pop('HF_HUB_OFFLINE', None)
    PREFLIGHT_ENV.pop('TRANSFORMERS_OFFLINE', None)
    subprocess.run([PY, '-u', str(SCRIPTS / 'apc_qwen_preflight.py'), str(CONFIG)], env=PREFLIGHT_ENV, cwd=REPO, check=True)

## Native dependencies, weights và renderer

Lần đầu có thể mất nhiều phút để build native và tải weights. Tiến độ gọn được in mỗi 20 giây;
log đầy đủ nằm trong `setup_logs/`. `setup_status.json` ghi rõ bước hỏng nếu cài đặt thất bại.
Chỉ chạy notebook này trong một phiên; không đồng thời chạy bản cũ dùng cùng WORK.

In [ ]:
with setup_stage('native_assets_renderer', OUT, CFG):
    'Select a CUDA toolkit for the cu124 PyTorch extensions before running native.py.'
    import json
    import os
    import re
    import shutil
    import subprocess
    import tarfile
    import tempfile
    import urllib.request
    from pathlib import Path
    import hashlib

    def is_cuda124(root):
        compiler = root / 'bin/nvcc'
        if not compiler.is_file() or not os.access(compiler, os.X_OK):
            return False
        version = subprocess.run([str(compiler), '--version'], capture_output=True, text=True)
        return version.returncode == 0 and re.search('release 12\\.4\\b', version.stdout) is not None

    def prepare_toolkit(work, env):
        private = work / 'cuda-12.4.1'
        roots = [Path('/usr/local/cuda-12.4'), Path('/usr/local/cuda')]
        current = shutil.which('nvcc', path=env.get('PATH'))
        if current:
            roots.append(Path(current).resolve().parent.parent)
        if (private / '.ready').exists():
            roots.append(private)
        for root in roots:
            if is_cuda124(root) and (root / 'include/cuda_runtime.h').exists() and (root / 'lib64/libcudart.so').exists():
                return root
        base = 'https://developer.download.nvidia.com/compute/cuda/redist/'
        with urllib.request.urlopen(base + 'redistrib_12.4.1.json', timeout=90) as response:
            manifest = json.load(response)
        private.mkdir(exist_ok=True)
        with tempfile.TemporaryDirectory(dir=work) as tmp:
            for package in ('cuda_nvcc', 'cuda_cudart', 'cuda_cccl'):
                item = manifest[package]['linux-x86_64']
                archive = Path(tmp) / (package + '.tar.xz')
                print('CUDA 12.4.1:', package, flush=True)
                digest = hashlib.sha256()
                with urllib.request.urlopen(base + item['relative_path'], timeout=180) as response, archive.open('wb') as dst:
                    for chunk in iter(lambda: response.read(1024 * 1024), b''):
                        digest.update(chunk)
                        dst.write(chunk)
                if digest.hexdigest() != item['sha256']:
                    raise RuntimeError('CUDA archive checksum mismatch: ' + package)
                unpack = Path(tmp) / package
                with tarfile.open(archive) as tar:
                    tar.extractall(unpack, filter='data')
                top, = unpack.iterdir()
                shutil.copytree(top, private, symlinks=True, dirs_exist_ok=True)
        if not (private / 'lib64').exists():
            (private / 'lib64').symlink_to('lib', target_is_directory=True)
        if not is_cuda124(private):
            raise RuntimeError('Private CUDA 12.4 compiler failed to start')
        (private / '.ready').touch()
        return private
    CUDA_WORK = Path('/kaggle/temp/apc_cuda_tools')
    CUDA_WORK.mkdir(parents=True, exist_ok=True)
    toolkit = prepare_toolkit(CUDA_WORK, ENV)
    ENV['CUDA_HOME'] = ENV['CUDA_PATH'] = str(toolkit)
    ENV['PATH'] = os.pathsep.join([str(toolkit / 'bin'), str(Path(PY).parent), ENV['PATH']])
    wheel_headers = sorted((Path(PY).parent.parent / 'lib/python3.10/site-packages/nvidia').glob('*/include'))
    ENV['CPATH'] = os.pathsep.join(filter(None, [str(toolkit / 'include'), *map(str, wheel_headers), ENV.get('CPATH')]))
    ENV['LIBRARY_PATH'] = os.pathsep.join(filter(None, [str(toolkit / 'lib64'), ENV.get('LIBRARY_PATH')]))
    ENV['LD_LIBRARY_PATH'] = os.pathsep.join(filter(None, [str(toolkit / 'lib64'), ENV.get('LD_LIBRARY_PATH')]))
    with tempfile.TemporaryDirectory(dir=WORK) as tmp:
        cu = Path(tmp) / 'probe.cu'
        cu.write_text('#include <cuda_runtime.h>\n__global__ void k() {}\nint main(){k<<<1,1>>>(); return cudaDeviceSynchronize()!=cudaSuccess;}')
        exe = Path(tmp) / 'probe'
        subprocess.run([str(toolkit / 'bin/nvcc'), '-arch=sm_75', str(cu), '-o', str(exe)], env=ENV, check=True)
        subprocess.run([str(exe)], env=ENV, check=True)
    print('CUDA 12.4 compile + launch PASS:', toolkit, flush=True)
    ENV.pop('HF_HUB_OFFLINE', None)
    ENV.pop('TRANSFORMERS_OFFLINE', None)
    run_logged('native_install', [PY, str(SCRIPTS / 'apc_install_native.py'), str(CONFIG)], OUT, env=ENV)
    run_logged('assets', [PY, str(SCRIPTS / 'apc_prepare_assets.py'), str(CONFIG), str(WORK / 'assets_ready.json')], OUT, env=ENV)
    assets = json.loads((WORK / 'assets_ready.json').read_text())
    ENV['PYTHONPATH'] += os.pathsep + str(REPO / 'apc/vision_modules/src/omni3d') + os.pathsep + str(REPO / 'apc/vision_modules/src/orient_anything')
    run_logged('compatibility_patches', [PY, str(SCRIPTS / 'apc_kaggle_patches.py'), str(REPO), assets['root'], str(WORK / 'patches.json')], OUT, env=ENV)
    ENV['HF_HUB_OFFLINE'] = '1'
    ENV['TRANSFORMERS_OFFLINE'] = '1'
    ENV['DISPLAY'] = ':99'
    ENV['LIBGL_ALWAYS_SOFTWARE'] = '1'
    ENV['MESA_GL_VERSION_OVERRIDE'] = '3.3'
    ENV['MPLBACKEND'] = 'Agg'
    if not Path('/tmp/.X99-lock').exists():
        display_log = (WORK / 'xvfb.log').open('a')
        XVFB = subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1280x1024x24', '-nolisten', 'tcp'], stdout=display_log, stderr=subprocess.STDOUT, env=ENV)
        time.sleep(2)
    probe = "\nimport io,json,torch,trimesh\nfrom PIL import Image\nfrom groundingdino import _C\nimport detectron2,pytorch3d,depth_pro,open3d,segment_anything\nfrom apc_kaggle_memory import plan_qwen_device_map\nfrom apc.apc_pipeline import APC\nscene=trimesh.Scene(trimesh.creation.box());raw=scene.save_image(resolution=(128,128),visible=False)\nim=Image.open(io.BytesIO(raw));im.verify()\nprint('Native imports + headless Trimesh render PASS',flush=True)\n"
    run_logged('native_renderer_probe', [PY, '-u', '-c', probe], OUT, env=ENV, cwd=REPO)

## Sửa viewpoint và chọn toàn bộ task của part

Chuẩn hóa object list dạng Python/JSON để dấu nháy không tạo scene key khác nhau.
Reference viewer được khớp duy nhất với đối tượng thật trong scene; chủ thể rõ của template câu hỏi được đối chiếu.
Không dùng ground truth để chọn người quan sát. Không giải quyết được sẽ ghi lỗi và giữ trace theo ngân sách.
Part 1 xen kẽ các cặp Left/Right và Visibility để kiểm tra sớm cả hai nhánh. Đủ mọi mẫu mới được tính là hoàn tất.

In [ ]:
with setup_stage('trace_reference_patch_and_data',OUT,CFG):
    run_logged('trace_patch',[PY,str(SCRIPTS/'apc_trace_patches.py'),str(REPO),str(WORK/'trace_patches.json')],OUT,env=ENV)
    run_logged('reference_patch',[PY,str(SCRIPTS/'apc_full_reference.py'),str(REPO),str(WORK/'reference_patches.json')],OUT,env=ENV)
    DATA_ENV=ENV.copy();DATA_ENV.pop('HF_HUB_OFFLINE',None);DATA_ENV.pop('TRANSFORMERS_OFFLINE',None)
    run_logged('prepare_data',[PY,str(SCRIPTS/'apc_prepare_data.py'),str(CONFIG)],OUT,env=DATA_ENV,cwd=REPO)
    manifest=json.loads((OUT/'manifest.json').read_text())
    assert manifest['selected_rows']==CFG['expected_rows'],manifest['selected_rows']
    assert manifest['circular_groups']==CFG['expected_circular_groups']
    assert manifest['tasks']==CFG['expected_task_counts']
    print('Selected:',manifest['tasks'])

## Tiếp tục benchmark
Worker bỏ qua mọi sample ID đã lưu và chỉ xử lý phần còn thiếu.

In [ ]:
from apc_launcher import launch as launch_full
def launch():
    return launch_full(CFG,PY,SCRIPTS,ENV,retry_failed=False)


## Chạy các lượt còn thiếu

In [ ]:
REPORT=launch()
print('Status:',REPORT['completion_status'],'technical:',REPORT['technical_status'])
print('Attempted:',REPORT['attempted_rows'],'/',REPORT['expected_rows'],'=',round(REPORT['attempted_percent'],2),'%')
print('Valid:',REPORT['valid_predictions'],'errors:',REPORT['pipeline_errors'],'invalid:',REPORT['invalid_outputs'])
print('Row accuracy:',REPORT['row_accuracy'],'Circular accuracy:',REPORT['circular_accuracy'])
print('Per task:',json.dumps(REPORT['by_task'],indent=2))
if REPORT['completion_status']=='PAUSED':
    print('Reached the worker time budget. Results so far are in Output.')
elif not REPORT['complete']:
    print('Inspect errors/coverage in full_report.json; accuracy keeps missing/error rows in denominator.')

## Lưu kết quả

`full_report.json` ghi coverage, lỗi và accuracy.
`results.zip`, `traces.zip` và `checkpoint.zip` lưu kết quả của phiên này.
Khi tổng hợp hai part, cộng số đúng và mẫu số.


In [ ]:
from apc_launcher import collect_reports
collect_reports(WORK,OUT)
try:
    packages=subprocess.check_output([str(UV),'pip','freeze','--python',PY],env=ENV,text=True)
    (OUT/'environment.txt').write_text(packages,encoding='utf-8')
except Exception as exc:
    print('Environment report:',repr(exc))
export_full(OUT,CFG)
show_downloads(OUT)
print('Download results and traces from this run. Check coverage before comparing accuracy.')

## Gộp benchmark, sau đó replay 120 mẫu
Kết quả replay không ghi đè đáp án benchmark. Nếu replay bị ngắt, lấy `trace_resume.zip` của phiên này, đặt `TRACE_RESUME_INPUT` và chạy lại notebook.

In [ ]:
from completion_helpers import merge_benchmark, prepare_replay, finish_replay
REFERENCE=Path('/kaggle/temp/apc_completion_reference.zip')
merge_benchmark(OUT,REFERENCE,Path('/kaggle/working/apc_completion/combined'))
if RUN_TRACE_REPLAY:
    TRACE_CFG=prepare_replay(CFG,OUT,REFERENCE,WORK,SCRIPTS,PY,DATA_ENV,TRACE_RESUME_INPUT)
    try:
        TRACE_REPORT=launch_full(TRACE_CFG,PY,SCRIPTS,ENV,retry_failed=False)
    finally:
        finish_replay(Path(TRACE_CFG['output']),REFERENCE)


## Tải kết quả
- `combined/benchmark_results.zip`: toàn bộ 2.078 dòng, giữ cả missing/error nếu chưa hoàn tất.
- `part1/checkpoint.zip`: dùng tiếp nếu benchmark vẫn còn thiếu.
- `trace_replay_120/trace_resume.zip`: metadata + predictions + trace; dùng tiếp khi replay bị ngắt.
- `trace_replay_120/replay_comparison.json`: đủ/thiếu trace, đáp án cũ/mới và số đúng/sai mới theo task.
- `trace_replay_120/traces.zip`: trace do exporter tạo nếu đủ ngân sách đĩa; `trace_resume.zip` cũng chứa trace đã giữ.

Không tải runtime/weights. Notebook không tự xóa dữ liệu benchmark để dành đĩa. Nếu gặp giới hạn tài nguyên, xem report và resume từ checkpoint mới nhất.
